In [ ]:
!pip install properscoring -q
!pip install properscoring openpyxl --quiet

In [ ]:
"""
UQ Experiment — Session 1 of 2
N_SIZES = [30, 50, 100]
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
import properscoring as ps
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ── Config ────────────────────────────────────────────────────
N_SIZES      = [30, 50, 100]           # SESSION 1
N_REPEATS    = 50
EPOCHS       = 500
LR           = 1e-3
LR_SGD       = 0.01
HIDDEN       = 64
DROPOUT_P    = 0.1
N_ENSEMBLE   = 5
MC_SAMPLES   = 50
SWAG_SAMPLES = 30
SWAG_START   = 0.60
BBB_KL_W     = 1e-4
COVERAGE     = 0.90
CAL_FRAC     = 0.20
TEST_FRAC    = 0.30
SIGMA_FLOOR  = 0.05
OUTPUT_FILE  = 'uq_results_s1.csv'
CKPT_FILE    = 'uq_checkpoint_s1.csv'

# ── Data ──────────────────────────────────────────────────────
def make_synthetic(n_total=1200, seed=42):
    rng   = np.random.RandomState(seed)
    X     = rng.randn(n_total, 8)
    w     = np.array([1.5, -2.0, 0.5, 1.0, -0.5, 0.3, -1.2, 0.8])
    noise = 0.3 + 0.5 * np.abs(X[:, 0])
    y     = X @ w + rng.randn(n_total) * noise
    return X.astype(np.float32), y.astype(np.float32)

def gaussian_nll(out, y_t):
    mu, lv = out[:, 0], out[:, 1]
    var = torch.exp(lv).clamp(1e-3, 1e3)
    return (0.5 * ((y_t - mu)**2 / var + lv)).mean()

# ── Models ────────────────────────────────────────────────────
class HeteroscedasticMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 2))
    def forward(self, x): return self.net(x)
    def predict_gaussian(self, x_np):
        self.eval()
        with torch.no_grad():
            out = self.forward(torch.tensor(x_np))
        mu    = out[:, 0].numpy()
        sigma = np.exp(0.5 * out[:, 1].numpy()).clip(1e-4, 1e2)
        return mu, sigma

class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        return nn.functional.linear(
            x, self.wm + ws * torch.randn_like(self.wm),
               self.bm + bs * torch.randn_like(self.bm))
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1  = BayesLinear(in_dim, hidden)
        self.l2  = BayesLinear(hidden, hidden)
        self.lo  = BayesLinear(hidden, 2)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    def kl(self): return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ── Training ──────────────────────────────────────────────────
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); gaussian_nll(model(Xt), yt).backward(); opt.step()

# ── Methods ───────────────────────────────────────────────────
def method_map(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_tr, y_tr)
    return model.predict_gaussian(X_te), None, None

def method_mcd(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te)) for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) + torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_ensemble(X_tr, y_tr, X_te):
    mus, sigsqs = [], []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = HeteroscedasticMLP(X_tr.shape[1])
        train_adam(model, X_tr, y_tr)
        m, s = model.predict_gaussian(X_te)
        mus.append(m); sigsqs.append(s**2)
    mus, sigsqs = np.array(mus), np.array(sigsqs)
    return (mus.mean(0), np.sqrt(mus.var(0) + sigsqs.mean(0))), None, None

def method_bbb(X_tr, y_tr, X_te):
    model = BayesMLP(X_tr.shape[1])
    opt   = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt) + BBB_KL_W * model.kl()
        loss.backward(); opt.step()
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te)) for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) + torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_swag(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    opt   = optim.SGD(model.parameters(), lr=LR_SGD,
                      momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=LR_SGD * 0.01)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []
    start  = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt)
        if torch.isnan(loss): break
        loss.backward(); opt.step(); scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten() for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    if len(w_list) < 2:
        return None, None, None
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    mus, sigsqs = [], []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape)); ptr += n
        mu, sig = model.predict_gaussian(X_te)
        sig = np.clip(sig, 1e-4, 5.0)
        mus.append(mu); sigsqs.append(sig**2)
    mus = np.array(mus)
    return (mus.mean(0), np.sqrt(mus.var(0) + np.array(sigsqs).mean(0))), None, None

def method_cp(X_tr, y_tr, X_te):
    n_cal = max(15, int(len(X_tr) * CAL_FRAC))
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal],  y_tr[:-n_cal]
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_t, y_t)
    mu_c, _ = model.predict_gaussian(X_c)
    alpha   = 1 - COVERAGE
    q       = np.quantile(np.abs(y_c - mu_c),
                          min(1.0, np.ceil((n_cal+1)*(1-alpha))/n_cal))
    mu_te, _ = model.predict_gaussian(X_te)
    z         = stats.norm.ppf(1 - alpha/2)
    return (mu_te, np.full(len(mu_te), q/z)), mu_te - q, mu_te + q

# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(y_true, mu, sigma, lower=None, upper=None):
    sigma_clipped = np.clip(sigma, SIGMA_FLOOR, None)
    crps    = float(ps.crps_gaussian(y_true, mu, sigma_clipped).mean())
    nll     = float(0.5 * np.mean(np.log(2*np.pi*sigma_clipped**2)
                                  + (y_true-mu)**2/sigma_clipped**2))
    sigma_raw = np.clip(sigma, 1e-4, None)
    nll_raw   = float(0.5 * np.mean(np.log(2*np.pi*sigma_raw**2)
                                    + (y_true-mu)**2/sigma_raw**2))
    alpha = 1 - COVERAGE
    z     = stats.norm.ppf(1 - alpha/2)
    lo    = lower if lower is not None else mu - z * sigma
    hi    = upper if upper is not None else mu + z * sigma
    picp  = float(np.mean((y_true >= lo) & (y_true <= hi)))
    mpiw  = float(np.mean(hi - lo))
    pen_lo   = (2/alpha) * (lo - y_true) * (y_true < lo)
    pen_hi   = (2/alpha) * (y_true - hi) * (y_true > hi)
    is_score = float(np.mean((hi - lo) + pen_lo + pen_hi))
    return {
        'crps':           round(crps, 4),
        'nll':            round(nll, 4),
        'nll_raw':        round(nll_raw, 4),
        'picp':           round(picp, 4),
        'mpiw':           round(mpiw, 4),
        'interval_score': round(is_score, 4),
        'sigma_mean':     round(float(sigma.mean()), 4),
    }

# ── Main ──────────────────────────────────────────────────────
def run():
    X, y = make_synthetic()
    sc_X, sc_y = StandardScaler(), StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    y = sc_y.fit_transform(y.reshape(-1,1)).ravel().astype(np.float32)
    n_test         = int(len(X) * TEST_FRAC)
    X_te, y_te     = X[-n_test:], y[-n_test:]
    X_pool, y_pool = X[:-n_test], y[:-n_test]

    methods = {
        'MAP':      method_map,
        'MCD':      method_mcd,
        'Ensemble': method_ensemble,
        'BBB':      method_bbb,
        'SWAG':     method_swag,
        'CP':       method_cp,
    }

    results = []
    t_start = time.time()

    for n in N_SIZES:
        print(f"\n>>> n = {n}", flush=True)
        for rep in range(N_REPEATS):
            idx            = np.random.RandomState(rep + n).choice(
                                 len(X_pool), n, replace=False)
            X_tr, y_tr     = X_pool[idx], y_pool[idx]

            for name, func in methods.items():
                t0 = time.time()
                try:
                    result = func(X_tr, y_tr, X_te)
                    if result[0] is None:          # SWAG diverged
                        results.append({
                            'method': name, 'n': n, 'rep': rep,
                            'time_s': round(time.time()-t0, 2),
                            'swag_ok': False,
                            'crps': np.nan, 'nll': np.nan,
                            'nll_raw': np.nan, 'picp': np.nan,
                            'mpiw': np.nan, 'interval_score': np.nan,
                            'sigma_mean': np.nan})
                        continue
                    (mu, sig), lo, hi = result
                    met = compute_metrics(y_te, mu, sig, lower=lo, upper=hi)
                    results.append({
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time()-t0, 2),
                        'swag_ok': True, **met})
                except Exception as e:
                    print(f"  ERROR {name} n={n} rep={rep}: {e}")

            # ── checkpoint every 10 reps ──────────────────────
            if (rep+1) % 10 == 0:
                elapsed = (time.time()-t_start)/60
                print(f"  rep {rep+1}/{N_REPEATS}  ({elapsed:.1f} min)",
                      flush=True)
                pd.DataFrame(results).to_csv(CKPT_FILE, index=False)

    return pd.DataFrame(results)

if __name__ == '__main__':
    t0 = time.time()
    df = run()
    df.to_csv(OUTPUT_FILE, index=False)

    print("\n" + "="*65)
    print(f"SESSION 1 COMPLETE  →  {OUTPUT_FILE}")
    print("="*65)

    for metric in ['crps', 'picp', 'interval_score']:
        label = " (target≈0.90)" if metric == 'picp' else " (↓ better)"
        print(f"\n── {metric.upper()}{label}")
        grp     = df.groupby(['method','n'])[metric]
        mean_df = grp.mean().unstack().round(3)
        std_df  = grp.std().unstack().round(3)
        for col in mean_df.columns:
            mean_df[col] = mean_df[col].astype(str) + " ±" + std_df[col].astype(str)
        print(mean_df.to_string())

    print("\n── SWAG convergence rate")
    sw = df[df.method=='SWAG'].groupby('n')['swag_ok'].agg(['sum','count'])
    sw['rate'] = (sw['sum']/sw['count']).round(2)
    print(sw.to_string())

    print(f"\nTotal runtime: {(time.time()-t0)/60:.1f} min")
    print(f"Rows saved: {len(df)}")
    print(f"\nNext: run Session 2 (n=200,500), then merge CSVs.")

In [ ]:
"""
UQ Experiment — Session 2 of 2
N_SIZES = [200, 500]
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
import properscoring as ps
from sklearn.preprocessing import StandardScaler
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ── Config ────────────────────────────────────────────────────
N_SIZES      = [200, 500]               # SESSION 2
N_REPEATS    = 50
EPOCHS       = 500
LR           = 1e-3
LR_SGD       = 0.01
HIDDEN       = 64
DROPOUT_P    = 0.1
N_ENSEMBLE   = 5
MC_SAMPLES   = 50
SWAG_SAMPLES = 30
SWAG_START   = 0.60
BBB_KL_W     = 1e-4
COVERAGE     = 0.90
CAL_FRAC     = 0.20
TEST_FRAC    = 0.30
SIGMA_FLOOR  = 0.05
OUTPUT_FILE  = 'uq_results_s2.csv'
CKPT_FILE    = 'uq_checkpoint_s2.csv'

# ── Data ──────────────────────────────────────────────────────
def make_synthetic(n_total=1200, seed=42):
    rng   = np.random.RandomState(seed)
    X     = rng.randn(n_total, 8)
    w     = np.array([1.5, -2.0, 0.5, 1.0, -0.5, 0.3, -1.2, 0.8])
    noise = 0.3 + 0.5 * np.abs(X[:, 0])
    y     = X @ w + rng.randn(n_total) * noise
    return X.astype(np.float32), y.astype(np.float32)

def gaussian_nll(out, y_t):
    mu, lv = out[:, 0], out[:, 1]
    var = torch.exp(lv).clamp(1e-3, 1e3)
    return (0.5 * ((y_t - mu)**2 / var + lv)).mean()

# ── Models ────────────────────────────────────────────────────
class HeteroscedasticMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 2))
    def forward(self, x): return self.net(x)
    def predict_gaussian(self, x_np):
        self.eval()
        with torch.no_grad():
            out = self.forward(torch.tensor(x_np))
        mu    = out[:, 0].numpy()
        sigma = np.exp(0.5 * out[:, 1].numpy()).clip(1e-4, 1e2)
        return mu, sigma

class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        return nn.functional.linear(
            x, self.wm + ws * torch.randn_like(self.wm),
               self.bm + bs * torch.randn_like(self.bm))
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1  = BayesLinear(in_dim, hidden)
        self.l2  = BayesLinear(hidden, hidden)
        self.lo  = BayesLinear(hidden, 2)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    def kl(self): return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ── Training ──────────────────────────────────────────────────
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); gaussian_nll(model(Xt), yt).backward(); opt.step()

# ── Methods ───────────────────────────────────────────────────
def method_map(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_tr, y_tr)
    return model.predict_gaussian(X_te), None, None

def method_mcd(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te)) for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) + torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_ensemble(X_tr, y_tr, X_te):
    mus, sigsqs = [], []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = HeteroscedasticMLP(X_tr.shape[1])
        train_adam(model, X_tr, y_tr)
        m, s = model.predict_gaussian(X_te)
        mus.append(m); sigsqs.append(s**2)
    mus, sigsqs = np.array(mus), np.array(sigsqs)
    return (mus.mean(0), np.sqrt(mus.var(0) + sigsqs.mean(0))), None, None

def method_bbb(X_tr, y_tr, X_te):
    model = BayesMLP(X_tr.shape[1])
    opt   = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt) + BBB_KL_W * model.kl()
        loss.backward(); opt.step()
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te)) for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) + torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_swag(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    opt   = optim.SGD(model.parameters(), lr=LR_SGD,
                      momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=LR_SGD * 0.01)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []
    start  = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt)
        if torch.isnan(loss): break
        loss.backward(); opt.step(); scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten() for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    if len(w_list) < 2:
        return None, None, None
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    mus, sigsqs = [], []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape)); ptr += n
        mu, sig = model.predict_gaussian(X_te)
        sig = np.clip(sig, 1e-4, 5.0)
        mus.append(mu); sigsqs.append(sig**2)
    mus = np.array(mus)
    return (mus.mean(0), np.sqrt(mus.var(0) + np.array(sigsqs).mean(0))), None, None

def method_cp(X_tr, y_tr, X_te):
    n_cal = max(15, int(len(X_tr) * CAL_FRAC))
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal],  y_tr[:-n_cal]
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_t, y_t)
    mu_c, _ = model.predict_gaussian(X_c)
    alpha   = 1 - COVERAGE
    q       = np.quantile(np.abs(y_c - mu_c),
                          min(1.0, np.ceil((n_cal+1)*(1-alpha))/n_cal))
    mu_te, _ = model.predict_gaussian(X_te)
    z         = stats.norm.ppf(1 - alpha/2)
    return (mu_te, np.full(len(mu_te), q/z)), mu_te - q, mu_te + q

# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(y_true, mu, sigma, lower=None, upper=None):
    sigma_clipped = np.clip(sigma, SIGMA_FLOOR, None)
    crps    = float(ps.crps_gaussian(y_true, mu, sigma_clipped).mean())
    nll     = float(0.5 * np.mean(np.log(2*np.pi*sigma_clipped**2)
                                  + (y_true-mu)**2/sigma_clipped**2))
    sigma_raw = np.clip(sigma, 1e-4, None)
    nll_raw   = float(0.5 * np.mean(np.log(2*np.pi*sigma_raw**2)
                                    + (y_true-mu)**2/sigma_raw**2))
    alpha = 1 - COVERAGE
    z     = stats.norm.ppf(1 - alpha/2)
    lo    = lower if lower is not None else mu - z * sigma
    hi    = upper if upper is not None else mu + z * sigma
    picp  = float(np.mean((y_true >= lo) & (y_true <= hi)))
    mpiw  = float(np.mean(hi - lo))
    pen_lo   = (2/alpha) * (lo - y_true) * (y_true < lo)
    pen_hi   = (2/alpha) * (y_true - hi) * (y_true > hi)
    is_score = float(np.mean((hi - lo) + pen_lo + pen_hi))
    return {
        'crps':           round(crps, 4),
        'nll':            round(nll, 4),
        'nll_raw':        round(nll_raw, 4),
        'picp':           round(picp, 4),
        'mpiw':           round(mpiw, 4),
        'interval_score': round(is_score, 4),
        'sigma_mean':     round(float(sigma.mean()), 4),
    }

# ── Main ──────────────────────────────────────────────────────
def run():
    X, y = make_synthetic()
    sc_X, sc_y = StandardScaler(), StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    y = sc_y.fit_transform(y.reshape(-1,1)).ravel().astype(np.float32)
    n_test         = int(len(X) * TEST_FRAC)
    X_te, y_te     = X[-n_test:], y[-n_test:]
    X_pool, y_pool = X[:-n_test], y[:-n_test]

    methods = {
        'MAP':      method_map,
        'MCD':      method_mcd,
        'Ensemble': method_ensemble,
        'BBB':      method_bbb,
        'SWAG':     method_swag,
        'CP':       method_cp,
    }

    results = []
    t_start = time.time()

    for n in N_SIZES:
        print(f"\n>>> n = {n}", flush=True)
        for rep in range(N_REPEATS):
            idx            = np.random.RandomState(rep + n).choice(
                                 len(X_pool), n, replace=False)
            X_tr, y_tr     = X_pool[idx], y_pool[idx]

            for name, func in methods.items():
                t0 = time.time()
                try:
                    result = func(X_tr, y_tr, X_te)
                    if result[0] is None:          # SWAG diverged
                        results.append({
                            'method': name, 'n': n, 'rep': rep,
                            'time_s': round(time.time()-t0, 2),
                            'swag_ok': False,
                            'crps': np.nan, 'nll': np.nan,
                            'nll_raw': np.nan, 'picp': np.nan,
                            'mpiw': np.nan, 'interval_score': np.nan,
                            'sigma_mean': np.nan})
                        continue
                    (mu, sig), lo, hi = result
                    met = compute_metrics(y_te, mu, sig, lower=lo, upper=hi)
                    results.append({
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time()-t0, 2),
                        'swag_ok': True, **met})
                except Exception as e:
                    print(f"  ERROR {name} n={n} rep={rep}: {e}")

            # ── checkpoint every 10 reps ──────────────────────
            if (rep+1) % 10 == 0:
                elapsed = (time.time()-t_start)/60
                print(f"  rep {rep+1}/{N_REPEATS}  ({elapsed:.1f} min)",
                      flush=True)
                pd.DataFrame(results).to_csv(CKPT_FILE, index=False)

    return pd.DataFrame(results)

if __name__ == '__main__':
    t0 = time.time()
    df = run()
    df.to_csv(OUTPUT_FILE, index=False)

    print("\n" + "="*65)
    print(f"SESSION 2 COMPLETE  →  {OUTPUT_FILE}")
    print("="*65)

    for metric in ['crps', 'picp', 'interval_score']:
        label = " (target≈0.90)" if metric == 'picp' else " (↓ better)"
        print(f"\n── {metric.upper()}{label}")
        grp     = df.groupby(['method','n'])[metric]
        mean_df = grp.mean().unstack().round(3)
        std_df  = grp.std().unstack().round(3)
        for col in mean_df.columns:
            mean_df[col] = mean_df[col].astype(str) + " ±" + std_df[col].astype(str)
        print(mean_df.to_string())

    print("\n── SWAG convergence rate")
    sw = df[df.method=='SWAG'].groupby('n')['swag_ok'].agg(['sum','count'])
    sw['rate'] = (sw['sum']/sw['count']).round(2)
    print(sw.to_string())

    print(f"\nTotal runtime: {(time.time()-t0)/60:.1f} min")
    print(f"Rows saved: {len(df)}")
    print(f"\nNext: merge with Session 1 CSV using the merge script.")

In [ ]:
"""
Merge Session 1 + Session 2 results into one CSV.
Run locally after downloading both CSVs from Kaggle.

Usage:
    python merge_sessions.py
"""
import pandas as pd

s1 = pd.read_csv('uq_results_s1.csv')
s2 = pd.read_csv('uq_results_s2.csv')

df = pd.concat([s1, s2], ignore_index=True)
df = df.sort_values(['n', 'method', 'rep']).reset_index(drop=True)

df.to_csv('uq_final_results.csv', index=False)

# Quick sanity check
print("=== Merge complete ===")
print(f"Session 1 rows : {len(s1)}")
print(f"Session 2 rows : {len(s2)}")
print(f"Total rows     : {len(df)}")
print(f"\nRows per (method, n):")
print(df.groupby(['method','n']).size().unstack().to_string())
print(f"\nSaved → uq_final_results.csv")

In [ ]:
"""
UQ Experiment — Real Datasets (UCI Energy + UCI Concrete)
=========================================================
Same pipeline as synthetic experiment.
Changes from synthetic version:
  1. load_data() fetches UCI datasets via sklearn / direct download
  2. N_SIZES capped at min(500, 70% of dataset size)
  3. OUTPUT_FILE and CKPT_FILE renamed
  4. Everything else identical
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
import properscoring as ps
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ── Config ────────────────────────────────────────────────────
DATASETS     = ['energy', 'concrete']   # run both
N_REPEATS    = 50
EPOCHS       = 500
LR           = 1e-3
LR_SGD       = 0.01
HIDDEN       = 64
DROPOUT_P    = 0.1
N_ENSEMBLE   = 5
MC_SAMPLES   = 50
SWAG_SAMPLES = 30
SWAG_START   = 0.60
BBB_KL_W     = 1e-4
COVERAGE     = 0.90
CAL_FRAC     = 0.20
TEST_FRAC    = 0.30
SIGMA_FLOOR  = 0.05

# n sizes will be set per dataset based on available pool size
N_SIZES_CANDIDATES = [30, 50, 100, 200, 500]

# ── Data loading ──────────────────────────────────────────────
def load_uci_energy():
    """
    UCI Energy Efficiency dataset.
    768 samples, 8 features, target: Heating Load (continuous).
    """
    try:
        data = fetch_openml(name='energy-efficiency', version=1,
                            as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        # Two targets: Y1 (heating) and Y2 (cooling) — use Y1
        y = data.target.astype(np.float32).values
        if y.ndim > 1:
            y = y[:, 0]
    except Exception:
        # Fallback: download directly from UCI
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "00242/ENB2012_data.xlsx")
        df = pd.read_excel(url)
        X = df.iloc[:, :8].values.astype(np.float32)
        y = df.iloc[:, 8].values.astype(np.float32)   # Y1: Heating Load
    print(f"  Energy: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.1f}, {y.max():.1f}]")
    return X, y

def load_uci_concrete():
    """
    UCI Concrete Compressive Strength dataset.
    1030 samples, 8 features, target: Compressive Strength (MPa).
    """
    try:
        data = fetch_openml(name='concrete_compressive_strength',
                            version=1, as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "concrete/compressive/Concrete_Data.xls")
        df = pd.read_excel(url)
        X = df.iloc[:, :8].values.astype(np.float32)
        y = df.iloc[:, 8].values.astype(np.float32)
    print(f"  Concrete: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.1f}, {y.max():.1f}]")
    return X, y

# ── Loss ──────────────────────────────────────────────────────
def gaussian_nll(out, y_t):
    mu, lv = out[:, 0], out[:, 1]
    var = torch.exp(lv).clamp(1e-3, 1e3)
    return (0.5 * ((y_t - mu)**2 / var + lv)).mean()

# ── Models ────────────────────────────────────────────────────
class HeteroscedasticMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 2))
    def forward(self, x): return self.net(x)
    def predict_gaussian(self, x_np):
        self.eval()
        with torch.no_grad():
            out = self.forward(torch.tensor(x_np))
        mu    = out[:, 0].numpy()
        sigma = np.exp(0.5 * out[:, 1].numpy()).clip(1e-4, 1e2)
        return mu, sigma

class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        return nn.functional.linear(
            x, self.wm + ws * torch.randn_like(self.wm),
               self.bm + bs * torch.randn_like(self.bm))
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1  = BayesLinear(in_dim, hidden)
        self.l2  = BayesLinear(hidden, hidden)
        self.lo  = BayesLinear(hidden, 2)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    def kl(self): return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ── Training ──────────────────────────────────────────────────
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); gaussian_nll(model(Xt), yt).backward(); opt.step()

# ── Methods ───────────────────────────────────────────────────
def method_map(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_tr, y_tr)
    return model.predict_gaussian(X_te), None, None

def method_mcd(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te))
                            for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) +
                       torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_ensemble(X_tr, y_tr, X_te):
    mus, sigsqs = [], []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = HeteroscedasticMLP(X_tr.shape[1])
        train_adam(model, X_tr, y_tr)
        m, s = model.predict_gaussian(X_te)
        mus.append(m); sigsqs.append(s**2)
    mus, sigsqs = np.array(mus), np.array(sigsqs)
    return (mus.mean(0), np.sqrt(mus.var(0) + sigsqs.mean(0))), None, None

def method_bbb(X_tr, y_tr, X_te):
    model = BayesMLP(X_tr.shape[1])
    opt   = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt) + BBB_KL_W * model.kl()
        loss.backward(); opt.step()
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te))
                            for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) +
                       torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_swag(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    opt   = optim.SGD(model.parameters(), lr=LR_SGD,
                      momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=LR_SGD * 0.01)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []; start = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt)
        if torch.isnan(loss): break
        loss.backward(); opt.step(); scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten()
                                for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    if len(w_list) < 2:
        return None, None, None
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    mus, sigsqs = [], []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape)); ptr += n
        mu, sig = model.predict_gaussian(X_te)
        sig = np.clip(sig, 1e-4, 5.0)
        mus.append(mu); sigsqs.append(sig**2)
    mus = np.array(mus)
    return (mus.mean(0), np.sqrt(mus.var(0) +
                                  np.array(sigsqs).mean(0))), None, None

def method_cp(X_tr, y_tr, X_te):
    n_cal = max(15, int(len(X_tr) * CAL_FRAC))
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal],  y_tr[:-n_cal]
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_t, y_t)
    mu_c, _ = model.predict_gaussian(X_c)
    alpha   = 1 - COVERAGE
    q       = np.quantile(np.abs(y_c - mu_c),
                          min(1.0, np.ceil((n_cal+1)*(1-alpha))/n_cal))
    mu_te, _ = model.predict_gaussian(X_te)
    z         = stats.norm.ppf(1 - alpha/2)
    return (mu_te, np.full(len(mu_te), q/z)), mu_te - q, mu_te + q

# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(y_true, mu, sigma, lower=None, upper=None):
    sigma_clipped = np.clip(sigma, SIGMA_FLOOR, None)
    crps    = float(ps.crps_gaussian(y_true, mu, sigma_clipped).mean())
    nll     = float(0.5 * np.mean(np.log(2*np.pi*sigma_clipped**2)
                                  + (y_true-mu)**2/sigma_clipped**2))
    sigma_raw = np.clip(sigma, 1e-4, None)
    nll_raw   = float(0.5 * np.mean(np.log(2*np.pi*sigma_raw**2)
                                    + (y_true-mu)**2/sigma_raw**2))
    alpha = 1 - COVERAGE
    z     = stats.norm.ppf(1 - alpha/2)
    lo    = lower if lower is not None else mu - z * sigma
    hi    = upper if upper is not None else mu + z * sigma
    picp  = float(np.mean((y_true >= lo) & (y_true <= hi)))
    mpiw  = float(np.mean(hi - lo))
    pen_lo   = (2/alpha) * (lo - y_true) * (y_true < lo)
    pen_hi   = (2/alpha) * (y_true - hi) * (y_true > hi)
    is_score = float(np.mean((hi - lo) + pen_lo + pen_hi))
    return {
        'crps':           round(crps, 4),
        'nll':            round(nll, 4),
        'nll_raw':        round(nll_raw, 4),
        'picp':           round(picp, 4),
        'mpiw':           round(mpiw, 4),
        'interval_score': round(is_score, 4),
        'sigma_mean':     round(float(sigma.mean()), 4),
    }

# ── Main experiment loop ──────────────────────────────────────
def run_dataset(dataset_name, X, y):
    print(f"\n{'='*55}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*55}")

    sc_X, sc_y = StandardScaler(), StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    y = sc_y.fit_transform(y.reshape(-1,1)).ravel().astype(np.float32)

    n_total = len(X)
    n_test  = int(n_total * TEST_FRAC)
    X_te, y_te     = X[-n_test:], y[-n_test:]
    X_pool, y_pool = X[:-n_test],  y[:-n_test]

    # Cap N_SIZES at pool size
    max_n  = int(len(X_pool) * 0.8)
    n_sizes = [n for n in N_SIZES_CANDIDATES if n <= max_n]
    print(f"  Total={n_total}, pool={len(X_pool)}, "
          f"test={n_test}, n_sizes={n_sizes}")

    methods = {
        'MAP':      method_map,
        'MCD':      method_mcd,
        'Ensemble': method_ensemble,
        'BBB':      method_bbb,
        'SWAG':     method_swag,
        'CP':       method_cp,
    }

    output_file = f"uq_results_{dataset_name}.csv"
    ckpt_file   = f"uq_checkpoint_{dataset_name}.csv"
    results     = []
    t_start     = time.time()

    for n in n_sizes:
        print(f"\n>>> n = {n}", flush=True)
        for rep in range(N_REPEATS):
            idx = np.random.RandomState(rep + n).choice(
                len(X_pool), n, replace=False)
            X_tr, y_tr = X_pool[idx], y_pool[idx]

            for name, func in methods.items():
                t0 = time.time()
                try:
                    result = func(X_tr, y_tr, X_te)
                    if result[0] is None:
                        results.append({
                            'dataset': dataset_name,
                            'method': name, 'n': n, 'rep': rep,
                            'time_s': round(time.time()-t0, 2),
                            'swag_ok': False,
                            'crps': np.nan, 'nll': np.nan,
                            'nll_raw': np.nan, 'picp': np.nan,
                            'mpiw': np.nan, 'interval_score': np.nan,
                            'sigma_mean': np.nan})
                        continue
                    (mu, sig), lo, hi = result
                    met = compute_metrics(y_te, mu, sig,
                                         lower=lo, upper=hi)
                    results.append({
                        'dataset': dataset_name,
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time()-t0, 2),
                        'swag_ok': True, **met})
                except Exception as e:
                    print(f"  ERROR {name} n={n} rep={rep}: {e}")

            if (rep+1) % 10 == 0:
                elapsed = (time.time()-t_start)/60
                print(f"  rep {rep+1}/{N_REPEATS}  ({elapsed:.1f} min)",
                      flush=True)
                pd.DataFrame(results).to_csv(ckpt_file, index=False)

    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False)

    # Quick summary
    print(f"\n── CRPS (↓ better) [{dataset_name}]")
    grp = df.groupby(['method','n'])['crps']
    mean_df = grp.mean().unstack().round(3)
    std_df  = grp.std().unstack().round(3)
    for col in mean_df.columns:
        mean_df[col] = (mean_df[col].astype(str) + " ±"
                        + std_df[col].astype(str))
    print(mean_df.to_string())

    print(f"\n── PICP (target 0.90) [{dataset_name}]")
    print(df.groupby(['method','n'])['picp'].mean().unstack().round(3)
            .to_string())

    print(f"\n── SWAG convergence [{dataset_name}]")
    sw = df[df.method=='SWAG'].groupby('n')['swag_ok'].agg(
        ['sum','count'])
    sw['rate'] = (sw['sum']/sw['count']).round(2)
    print(sw.to_string())

    print(f"\nSaved → {output_file}")
    return df

# ── Run both datasets ─────────────────────────────────────────
if __name__ == '__main__':
    all_results = []
    t0_total = time.time()

    print("Loading UCI datasets...")
    datasets = {}
    try:
        datasets['energy']   = load_uci_energy()
    except Exception as e:
        print(f"  Energy load failed: {e}")
    try:
        datasets['concrete'] = load_uci_concrete()
    except Exception as e:
        print(f"  Concrete load failed: {e}")

    for name, (X, y) in datasets.items():
        df = run_dataset(name, X, y)
        all_results.append(df)

    if all_results:
        combined = pd.concat(all_results, ignore_index=True)
        combined.to_csv('uq_results_real_combined.csv', index=False)
        print(f"\nCombined saved → uq_results_real_combined.csv")

    print(f"\nTotal runtime: {(time.time()-t0_total)/60:.1f} min")
    print("\nNext: merge with synthetic results for joint BHM analysis")

In [ ]:
# 1 R 代码复制整理 手动打勾
# 2 绘图代码整理   手动打勾
# 3 realdataset 结果整理添加
Qishi


# 1 美化figure
# paper structure 检查
# SWAG 数据小的时候不收敛 NLL也有爆炸的问题 
Minxuan

In [ ]:
# R code, 需要先下载我的output, uq_final_results.csv
library(tidyverse)
library(brms)
library(tidybayes)

set.seed(42)

CSV_PATH <- "uq_final_results.csv"
OUT_DIR  <- "processed_data" 
dir.create(OUT_DIR, showWarnings = FALSE)

# 1. Load & clean
raw <- read_csv(CSV_PATH, show_col_types = FALSE)

df <- raw %>%
  filter(!(method == "SWAG" & swag_ok == FALSE)) %>%
  mutate(
    method = factor(method, levels = c("MAP", "MCD", "Ensemble", "SWAG", "BBB", "CP")),
    n_num  = as.integer(as.character(n)),
    n      = factor(n)
  )

# 2. Metric stability
stability_all <- df %>%
  select(method, n_num, rep, crps, nll, picp, mpiw, interval_score) %>%
  pivot_longer(c(crps, nll, picp, mpiw, interval_score), names_to = "metric", values_to = "value") %>%
  filter(!is.na(value)) %>%
  group_by(method, metric, n_num) %>%
  summarise(
    mean_val = mean(value), 
    sd_val   = pmax(sd(value), 1e-5), 
    .groups  = "drop"
  )

write_csv(stability_all, file.path(OUT_DIR, "exp1_metric_stability.csv"))

# 3. BHM Pairwise Comparison & MDD
TARGET_METRICS <- c("crps", "picp", "interval_score")
N_VALS         <- sort(unique(df$n_num))
bhm_results    <- list()

for (metric in TARGET_METRICS) {
  for (n_val in N_VALS) {
    d_sub <- df %>%
      filter(n_num == n_val, !is.na(.data[[metric]])) %>%
      select(method, value = all_of(metric))
      
    if (nrow(d_sub) < 10) next

    fit <- tryCatch({
      brm(value ~ 0 + method, data = d_sub,
          prior = c(prior(normal(0, 1), class = b), prior(exponential(1), class = sigma)),
          chains = 2, iter = 2000, warmup = 1000, cores = 2, refresh = 0, silent = 2)
    }, error = function(e) { NULL })

    if (is.null(fit)) next

    draws <- as_draws_df(fit)
    method_names <- gsub("^b_method", "", names(draws)[grepl("^b_method", names(draws))])

    pairs <- expand_grid(A = method_names, B = method_names) %>%
      filter(A != B) %>%
      rowwise() %>%
      mutate(
        col_A = paste0("b_method", A), col_B = paste0("b_method", B),
        prob_A_better = mean(draws[[col_A]] < draws[[col_B]]),
        mdd_80        = 1.28 * sd(draws[[col_A]] - draws[[col_B]]) 
      ) %>%
      ungroup() %>%
      mutate(metric = metric, n = n_val)

    bhm_results[[paste(metric, n_val, sep = "_")]] <- pairs
  }
}
bhm_df <- bind_rows(bhm_results)

write_csv(bhm_df, file.path(OUT_DIR, "exp2_3_bhm_pairwise_probs_and_mdd.csv"))

# 4. Cross-metric rank consistency
rank_consistency <- df %>%
  filter(!is.na(crps), !is.na(interval_score)) %>%
  group_by(n_num, rep) %>%
  summarise(
    tau = cor(rank(crps), rank(interval_score), method = "kendall"), 
    .groups = "drop"
  ) %>%
  mutate(n_factor = factor(n_num))

write_csv(rank_consistency, file.path(OUT_DIR, "exp4_rank_consistency.csv"))

# 5. Tables
tbl1 <- df %>%
  group_by(method, n_num) %>%
  summarise(
    CRPS = sprintf("%.3f ± %.3f", mean(crps, na.rm=T), sd(crps, na.rm=T)),
    NLL  = sprintf("%.2f ± %.2f", mean(nll, na.rm=T), sd(nll, na.rm=T)),
    PICP = sprintf("%.3f ± %.3f", mean(picp, na.rm=T), sd(picp, na.rm=T)),
    IS   = sprintf("%.3f ± %.3f", mean(interval_score, na.rm=T), sd(interval_score, na.rm=T)),
    .groups = "drop"
  ) %>%
  arrange(n_num, method)

write_csv(tbl1, file.path(OUT_DIR, "table1_mean_sd_results.csv"))

if (nrow(bhm_df) > 0) {
  tbl2 <- bhm_df %>%
    filter(metric == "crps", B == "MAP") %>%
    select(A, n, prob_A_better) %>%
    pivot_wider(names_from = n, values_from = prob_A_better, names_prefix = "n=") %>%
    rename(Method = A) %>%
    mutate(across(where(is.numeric), ~ round(., 3)))
    
  write_csv(tbl2, file.path(OUT_DIR, "table2_prob_beats_map.csv"))
}

In [ ]:
# 画图代码
# ═══════════════════════════════════════════════════════════════
# Input:  uq_final_results.csv, bhm_pairwise_probs.csv
# Output: figures/ (PDF + PNG)
# ═══════════════════════════════════════════════════════════════
# install.packages(c("ggdist", "RColorBrewer", "scales", "tidyverse", "patchwork"))

library(tidyverse)
library(ggdist)
library(RColorBrewer)
library(patchwork)

set.seed(42)
dir.create("figures", showWarnings = FALSE)

N_VALS <- c(30, 50, 100, 200, 500)

PALETTE <- c(
  MCD      = "#1f77b4",
  Ensemble = "#d62728",
  BBB      = "#2ca02c",
  SWAG     = "#ff7f0e",
  CP       = "#7f7f7f",
  MAP      = "#9467bd"
)

theme_pub <- function() {
  theme_bw(base_size = 12) %+replace% theme(
    panel.grid.minor   = element_blank(),
    panel.grid.major   = element_line(color = "grey90", linewidth = 0.4),
    panel.border       = element_rect(color = "black", fill = NA,
                                      linewidth = 0.8),
    axis.text          = element_text(color = "black", size = 10),
    axis.title         = element_text(face = "bold", size = 12),
    plot.title         = element_text(face = "bold", size = 13,
                                      hjust = 0.5, margin = margin(b = 8)),
    strip.background   = element_rect(fill = "grey95", color = "black",
                                      linewidth = 0.8),
    strip.text         = element_text(face = "bold", size = 11),
    legend.position    = "right"
  )
}

save_fig <- function(p, name, w = 7, h = 4.5) {
  ggsave(paste0("figures/", name, ".pdf"), p, width = w, height = h)
  ggsave(paste0("figures/", name, ".png"), p, width = w, height = h,
         dpi = 150)
  cat(sprintf("  Saved %s\n", name))
}

# ── Load data ────────────────────────────────────────────────
cat("Loading data...\n")
bhm <- read_csv("bhm_pairwise_probs.csv", show_col_types = FALSE)

raw <- read_csv("uq_final_results.csv", show_col_types = FALSE)
df  <- raw %>%
  filter(!(method == "SWAG" & swag_ok == FALSE)) %>%
  mutate(
    method = factor(method, levels = names(PALETTE)),
    n_num  = as.integer(as.character(n))
  )

summary_df <- df %>%
  group_by(method, n_num) %>%
  summarise(
    crps_mean = mean(crps, na.rm = TRUE),
    crps_sd   = sd(crps,   na.rm = TRUE),
    picp_mean = mean(picp, na.rm = TRUE),
    picp_sd   = sd(picp,   na.rm = TRUE),
    is_mean   = mean(interval_score, na.rm = TRUE),
    is_sd     = sd(interval_score,   na.rm = TRUE),
    .groups   = "drop"
  )

# ── Fig 1: CRPS mean ± SD ───────────────────────────────────
cat("Drawing Fig 1...\n")

df_ribbon <- summary_df %>% filter(method != "SWAG")
df_swag   <- summary_df %>% filter(method == "SWAG")

fig1 <- ggplot(mapping = aes(x = n_num)) +
  geom_ribbon(
    data = df_ribbon,
    aes(ymin  = pmax(crps_mean - crps_sd, 0),
        ymax  = crps_mean + crps_sd,
        fill  = method, group = method),
    alpha = 0.13, color = NA
  ) +
  geom_line(
    data = df_ribbon,
    aes(y = crps_mean, color = method, group = method),
    linewidth = 1.1
  ) +
  geom_point(
    data = df_ribbon,
    aes(y = crps_mean, color = method),
    shape = 21, size = 3, stroke = 1.2, fill = "white"
  ) +
  geom_line(
    data = df_swag,
    aes(y = crps_mean, color = method, group = method),
    linewidth = 1.1, linetype = "dashed"
  ) +
  geom_point(
    data = df_swag,
    aes(y = crps_mean, color = method),
    shape = 21, size = 3, stroke = 1.2, fill = "white"
  ) +
  scale_x_log10(breaks = N_VALS, labels = N_VALS) +
  scale_y_continuous(limits = c(0.10, 0.45)) +
  scale_color_manual(values = PALETTE) +
  scale_fill_manual(values  = PALETTE) +
  annotate("text", x = 35, y = 0.44,
           label = "SWAG shown as dashed line (SD omitted; high variance at small n)",
           hjust = 0, size = 3.2, color = "grey40") +
  labs(
    title = "Mean CRPS across Training Sizes",
    x     = "Training Size n  (log scale)",
    y     = "Mean CRPS  (lower is better)",
    color = "Method", fill = "Method"
  ) +
  theme_pub()

save_fig(fig1, "fig1_crps_mean_ci")

# ── Fig 2: All metrics SD (log-log facet) ───────────────────
cat("Drawing Fig 2...\n")

stability_df <- df %>%
  group_by(method, n_num) %>%
  summarise(
    CRPS             = sd(crps,           na.rm = TRUE),
    PICP             = sd(picp,           na.rm = TRUE),
    MPIW             = sd(mpiw,           na.rm = TRUE),
    `Interval Score` = sd(interval_score, na.rm = TRUE),
    .groups = "drop"
  ) %>%
  pivot_longer(c(CRPS, PICP, MPIW, `Interval Score`),
               names_to = "metric", values_to = "sd_val") %>%
  mutate(metric = factor(metric,
                         levels = c("CRPS","Interval Score","PICP","MPIW")))

fig2 <- stability_df %>%
  ggplot(aes(x = n_num, y = sd_val, color = method, group = method)) +
  geom_line(linewidth = 1) +
  geom_point(shape = 21, size = 2.5, stroke = 1.1, fill = "white") +
  scale_x_log10(breaks = N_VALS, labels = N_VALS) +
  scale_y_log10() +
  scale_color_manual(values = PALETTE) +
  facet_wrap(~ metric, scales = "free_y", nrow = 2) +
  labs(
    title = "Metric Estimation Volatility",
    x     = "Training Size n  (log scale)",
    y     = "Standard Deviation  (log scale)",
    color = "Method"
  ) +
  theme_pub() +
  theme(legend.position = "bottom")

save_fig(fig2, "fig2_all_metrics_stability", w = 8, h = 6)

# ── Fig 3: P(MCD < Ensemble) ────────────────────────────────
cat("Drawing Fig 3...\n")

fig3_data <- bhm %>%
  filter(metric == "crps", A == "MCD", B == "Ensemble") %>%
  mutate(n = as.integer(n))

fig3 <- ggplot(fig3_data, aes(x = n, y = prob_A_better)) +
  annotate("rect",
           xmin = 25, xmax = 600,
           ymin = 0.95, ymax = 1.01,
           fill = "#e5f5e0", alpha = 0.7) +
  geom_hline(yintercept = 0.95, linetype = "dashed",
             color = "#d62728", linewidth = 1) +
  geom_hline(yintercept = 0.50, linetype = "dotted",
             color = "grey50", linewidth = 0.8) +
  geom_line(linewidth = 1.3, color = PALETTE["MCD"]) +
  geom_point(shape = 21, size = 4, stroke = 1.5,
             fill = "white", color = PALETTE["MCD"]) +
  annotate("text", x = 55, y = 0.905,
           label = "Significance Threshold (P > 0.95)",
           color = "#d62728", fontface = "italic",
           hjust = 0, size = 3.5) +
  annotate("text", x = 55, y = 0.455,
           label = "Random Guessing (P = 0.50)",
           color = "grey40", hjust = 0, size = 3.5) +
  scale_x_log10(breaks = N_VALS, labels = N_VALS) +
  scale_y_continuous(limits = c(0, 1.01),
                     labels = scales::percent_format(accuracy = 1)) +
  labs(
    title = "Ranking Stability: MCD vs. Deep Ensemble",
    x     = "Training Size n",
    y     = "Posterior Probability P(MCD < Ensemble)"
  ) +
  theme_pub()

save_fig(fig3, "fig3_ranking_stability")

# ── Fig 4: All methods vs Ensemble ──────────────────────────
cat("Drawing Fig 4...\n")

fig4_data <- bhm %>%
  filter(metric == "crps", B == "Ensemble", A != "Ensemble") %>%
  mutate(
    n = factor(as.integer(n), levels = N_VALS),
    A = factor(A, levels = names(PALETTE))
  )

fig4 <- ggplot(fig4_data,
               aes(x = n, y = prob_A_better, color = A, group = A)) +
  geom_hline(yintercept = 0.50, linetype = "dotted",
             color = "grey50", linewidth = 0.8) +
  geom_hline(yintercept = 0.95, linetype = "dashed",
             color = "#d62728", linewidth = 0.8) +
  geom_line(linewidth = 1.1) +
  geom_point(shape = 21, size = 3, stroke = 1.2, fill = "white") +
  scale_y_continuous(limits = c(0, 1),
                     labels = scales::percent_format(accuracy = 1)) +
  scale_color_manual(values = PALETTE) +
  facet_wrap(~ A, nrow = 1) +
  labs(
    title = "Probability of Outperforming Deep Ensemble on CRPS",
    x     = "Training Size n",
    y     = "P(Method < Ensemble)"
  ) +
  theme_pub() +
  theme(legend.position = "none",
        axis.text.x     = element_text(size = 8))

save_fig(fig4, "fig4_vs_ensemble", w = 11, h = 4)

# ── Fig 5: MDD curves across metrics ────────────────────────
cat("Drawing Fig 5...\n")

METRIC_COLORS <- c(
  CRPS             = "#1f77b4",
  `Interval Score` = "#d62728",
  PICP             = "#2ca02c"
)

fig5_data <- bhm %>%
  filter(A == "MCD", B == "Ensemble",
         metric %in% c("crps", "interval_score", "picp")) %>%
  mutate(
    n = as.integer(n),
    metric_label = factor(
      recode(metric,
             crps = "CRPS",
             interval_score = "Interval Score",
             picp = "PICP"),
      levels = c("CRPS", "Interval Score", "PICP")),
    anomalous = (n == 50)
  )

fig5 <- ggplot(fig5_data,
               aes(x        = factor(n, levels = N_VALS),
                   y        = mdd_80,
                   color    = metric_label,
                   linetype = metric_label,
                   group    = metric_label)) +
  geom_line(linewidth = 1.2,
            position  = position_dodge(0.15)) +
  geom_point(data     = filter(fig5_data, !anomalous),
             shape    = 21, size = 3.5, stroke = 1.3,
             fill     = "white",
             position = position_dodge(0.15)) +
  geom_point(data     = filter(fig5_data, anomalous),
             shape    = 4, size = 4.5, stroke = 1.8,
             position = position_dodge(0.15)) +
  annotate("text", x = "50", y = 2.5,
           label = "× unreliable\n(n=50 BHM)",
           color = "grey40", size = 3, hjust = 0.5) +
  scale_y_log10(
    labels = function(x) ifelse(x >= 0.01,
                                sprintf("%.2f", x),
                                sprintf("%.3f", x))
  ) +
  scale_color_manual(values   = METRIC_COLORS) +
  scale_linetype_manual(values = c("solid", "dashed", "dotted")) +
  labs(
    title    = "Minimum Detectable Difference: MCD vs. Deep Ensemble",
    x        = "Training Size n",
    y        = "MDD  (log scale, 80% power)",
    color    = NULL, linetype = NULL
  ) +
  theme_pub() +
  theme(
    legend.position      = c(0.97, 0.97),
    legend.justification = c(1, 1),
    legend.background    = element_rect(fill = "white", color = "grey70"),
    legend.text          = element_text(size = 10)
  )

save_fig(fig5, "fig5_mdd_curves", w = 7, h = 5)

# ── Fig 6: Cross-metric rank consistency (raincloud) ────────
cat("Drawing Fig 6...\n")

rank_df <- df %>%
  filter(!is.na(crps), !is.na(interval_score)) %>%
  group_by(n_num, rep) %>%
  summarise(
    tau = cor(rank(crps), rank(interval_score), method = "kendall"),
    .groups = "drop"
  ) %>%
  mutate(n_factor = factor(n_num, levels = N_VALS))

blues <- colorRampPalette(brewer.pal(9, "Blues")[3:8])(5)

fig6 <- ggplot(rank_df,
               aes(x = n_factor, y = tau,
                   fill = n_factor, color = n_factor)) +
  stat_halfeye(
    adjust        = 2,
    width         = 0.55,
    justification = -0.18,
    .width        = 0,
    point_colour  = NA,
    alpha         = 0.75,
    color         = NA
  ) +
  geom_boxplot(
    width         = 0.15,
    outlier.shape = 21, outlier.size  = 1.8,
    outlier.fill  = "white",
    fill          = "white", alpha = 0.9, linewidth = 0.7
  ) +
  geom_hline(yintercept = 1, linetype = "dashed",
             color = "#d62728", linewidth = 1) +
  scale_fill_manual(values  = blues) +
  scale_color_manual(values = blues) +
  coord_cartesian(ylim = c(min(rank_df$tau) - 0.05, 1.05)) +
  labs(
    title = "Cross-Metric Rank Consistency (CRPS vs. Interval Score)",
    x     = "Training Size n",
    y     = "Kendall's \u03c4"
  ) +
  theme_pub() +
  theme(legend.position = "none")

ggsave("figures/fig6_rank_consistency.pdf", fig6,
       width = 8, height = 5, device = cairo_pdf)
ggsave("figures/fig6_rank_consistency.png", fig6,
       width = 8, height = 5, dpi = 150)
cat("  Saved fig6_rank_consistency\n")

# ── Power-law alpha ──────────────────────────────────────────
cat("\nCRPS power-law alpha (Var ~ n^{-alpha}):\n")
stability_df %>%
  filter(metric == "CRPS", sd_val > 0) %>%
  mutate(log_n = log(n_num), log_sd = log(sd_val)) %>%
  group_by(method) %>%
  summarise(
    alpha = -coef(lm(log_sd ~ log_n))[["log_n"]],
    .groups = "drop"
  ) %>%
  arrange(desc(alpha)) %>%
  mutate(alpha = round(alpha, 2)) %>%
  print()


In [ ]:
"""
UQ Experiment — Classification (UCI Breast Cancer)
===================================================
Binary classification version of the BDL benchmark.
Methods: MAP, MCD, Ensemble, BBB, SWAG, Conformal Prediction
Metrics: Brier Score, NLL (BCE), ECE
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
from sklearn.calibration import calibration_curve
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
DATASET      = 'breast_cancer'   # UCI Breast Cancer Wisconsin
N_SIZES      = [30, 50, 100, 200]  # pool=398, max_n≈320
N_REPEATS    = 50
EPOCHS       = 500
LR           = 1e-3
HIDDEN       = 64
DROPOUT_P    = 0.1
N_ENSEMBLE   = 5
MC_SAMPLES   = 50
SWAG_SAMPLES = 30
SWAG_START   = 0.60
BBB_KL_W     = 1e-4
COVERAGE     = 0.90           # For conformal prediction
CAL_FRAC     = 0.20           # Calibration split for CP
TEST_FRAC    = 0.30

# ══════════════════════════════════════════════════════════════
# DATA LOADING
# ══════════════════════════════════════════════════════════════
def load_breast_cancer_data():
    """
    UCI Breast Cancer Wisconsin dataset.
    569 samples, 30 features, binary classification (malignant=1, benign=0).
    """
    data = load_breast_cancer()
    X = data.data.astype(np.float32)
    y = data.target.astype(np.float32)
    print(f"  Breast Cancer: X={X.shape}, y={y.shape}, "
          f"class balance: {y.mean():.3f} (malignant proportion)")
    return X, y

# ══════════════════════════════════════════════════════════════
# LOSS FUNCTION
# ══════════════════════════════════════════════════════════════
def binary_cross_entropy(out, y_t):
    """Binary cross entropy loss with logits input."""
    return nn.functional.binary_cross_entropy_with_logits(out.squeeze(), y_t)

# ══════════════════════════════════════════════════════════════
# MODELS
# ══════════════════════════════════════════════════════════════
class BinaryMLP(nn.Module):
    """2-hidden-layer MLP for binary classification."""
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 1)   # logits output (no sigmoid)
        )
    
    def forward(self, x):
        return self.net(x)
    
    def predict_proba(self, x_np):
        """Return probability of class 1."""
        self.eval()
        with torch.no_grad():
            logits = self.forward(torch.tensor(x_np))
        proba = torch.sigmoid(logits).squeeze().numpy()
        return proba

class BayesLinear(nn.Module):
    """Mean-field Gaussian Bayesian linear layer for classification."""
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        w = self.wm + ws * torch.randn_like(self.wm)
        b = self.bm + bs * torch.randn_like(self.bm)
        return nn.functional.linear(x, w, b)
    
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesBinaryMLP(nn.Module):
    """Bayesian MLP for binary classification."""
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1 = BayesLinear(in_dim, hidden)
        self.l2 = BayesLinear(hidden, hidden)
        self.lo = BayesLinear(hidden, 1)   # output logits
        self.act = nn.ReLU()
    
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    
    def kl(self):
        return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ══════════════════════════════════════════════════════════════
# TRAINING
# ══════════════════════════════════════════════════════════════
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = binary_cross_entropy(model(Xt), yt)
        loss.backward()
        opt.step()

# ══════════════════════════════════════════════════════════════
# METHODS
# ══════════════════════════════════════════════════════════════
def method_map(X_tr, y_tr, X_te):
    model = BinaryMLP(X_tr.shape[1])
    train_adam(model, X_tr, y_tr)
    proba = model.predict_proba(X_te)
    return proba, None, None

def method_mcd(X_tr, y_tr, X_te):
    model = BinaryMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        logits = torch.stack([model(torch.tensor(X_te)) 
                              for _ in range(MC_SAMPLES)])
    proba = torch.sigmoid(logits).squeeze().numpy()
    proba_mean = proba.mean(axis=0)
    return proba_mean, None, None

def method_ensemble(X_tr, y_tr, X_te):
    probas = []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = BinaryMLP(X_tr.shape[1])
        train_adam(model, X_tr, y_tr)
        probas.append(model.predict_proba(X_te))
    probas = np.array(probas)
    return probas.mean(axis=0), None, None

def method_bbb(X_tr, y_tr, X_te):
    model = BayesBinaryMLP(X_tr.shape[1])
    opt = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = binary_cross_entropy(model(Xt), yt) + BBB_KL_W * model.kl()
        loss.backward()
        opt.step()
    model.train()
    with torch.no_grad():
        logits = torch.stack([model(torch.tensor(X_te)) 
                              for _ in range(MC_SAMPLES)])
    proba = torch.sigmoid(logits).squeeze().numpy()
    return proba.mean(axis=0), None, None

def method_swag(X_tr, y_tr, X_te):
    model = BinaryMLP(X_tr.shape[1])
    opt = optim.SGD(model.parameters(), lr=LR * 10, momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []
    start = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = binary_cross_entropy(model(Xt), yt)
        if torch.isnan(loss):
            break
        loss.backward()
        opt.step()
        scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten() for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    if len(w_list) < 2:
        return None, None, None
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    probas = []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape))
            ptr += n
        probas.append(model.predict_proba(X_te))
    probas = np.array(probas)
    return probas.mean(axis=0), None, None

def method_cp(X_tr, y_tr, X_te):
    """
    Split Conformal Prediction for binary classification.
    Returns prediction sets (intervals) as {0}, {1}, or {0,1}.
    """
    n_cal = max(15, int(len(X_tr) * CAL_FRAC))
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal], y_tr[:-n_cal]
    
    model = BinaryMLP(X_tr.shape[1])
    train_adam(model, X_t, y_t)
    
    # Get calibration scores (non-conformity scores)
    proba_c = model.predict_proba(X_c)
    # Non-conformity: 1 - probability of true class
    scores = 1 - (proba_c * y_c + (1 - proba_c) * (1 - y_c))
    
    # Compute quantile
    alpha = 1 - COVERAGE
    q = np.quantile(scores, np.ceil((n_cal + 1) * (1 - alpha)) / (n_cal + 1))
    
    # Prediction sets for test
    proba_te = model.predict_proba(X_te)
    # Prediction set: include class 1 if 1 - proba <= q (i.e., proba >= 1 - q)
    # Include class 0 if proba <= q
    pred_set_contains_1 = (1 - proba_te) <= q
    pred_set_contains_0 = proba_te <= q
    
    # For metrics, we need a point prediction and "interval" representation
    # Use the most confident class as point prediction
    point_pred = (proba_te >= 0.5).astype(np.float32)
    
    # For ECE calculation, use the probability
    return point_pred, proba_te, (pred_set_contains_0, pred_set_contains_1)

# ══════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════
def compute_brier_score(y_true, proba):
    """Brier Score: mean squared error between predicted probability and outcome."""
    return float(np.mean((proba - y_true) ** 2))

def compute_ece(y_true, proba, n_bins=10):
    """
    Expected Calibration Error (ECE).
    Bins predictions into n_bins and computes weighted average of calibration gaps.
    """
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        in_bin = (proba >= bin_lower) & (proba < bin_upper)
        if bin_upper == 1.0:
            in_bin = (proba >= bin_lower) & (proba <= bin_upper)
        if not np.any(in_bin):
            continue
        prop_in_bin = np.mean(in_bin)
        acc_in_bin = np.mean(y_true[in_bin])
        conf_in_bin = np.mean(proba[in_bin])
        ece += prop_in_bin * np.abs(acc_in_bin - conf_in_bin)
    return float(ece)

def compute_metrics(y_true, proba, pred_set_info=None):
    """
    Compute Brier Score, NLL (BCE), ECE.
    For conformal prediction, also compute coverage and set size.
    """
    # Brier Score
    brier = compute_brier_score(y_true, proba)
    
    # Negative Log-Likelihood (Binary Cross Entropy)
    nll = float(-np.mean(y_true * np.log(proba + 1e-10) + 
                         (1 - y_true) * np.log(1 - proba + 1e-10)))
    
    # Expected Calibration Error
    ece = compute_ece(y_true, proba)
    
    metrics = {
        'brier': round(brier, 4),
        'nll': round(nll, 4),
        'ece': round(ece, 4),
    }
    
    # Conformal prediction metrics (if provided)
    if pred_set_info is not None:
        contains_0, contains_1 = pred_set_info
        coverage = np.mean((y_true == 0) & contains_0 | (y_true == 1) & contains_1)
        set_size = np.mean(contains_0.astype(int) + contains_1.astype(int))
        metrics['cp_coverage'] = round(coverage, 4)
        metrics['cp_set_size'] = round(set_size, 4)
    
    return metrics

# ══════════════════════════════════════════════════════════════
# MAIN EXPERIMENT LOOP
# ══════════════════════════════════════════════════════════════
def run_experiment():
    print("\n" + "=" * 60)
    print("Classification UQ Benchmark - UCI Breast Cancer")
    print("=" * 60)
    
    # Load data
    X, y = load_breast_cancer_data()
    
    # Standardize features
    sc_X = StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    
    # Fixed test split
    n_test = int(len(X) * TEST_FRAC)
    X_te = X[-n_test:]
    y_te = y[-n_test:]
    X_pool = X[:-n_test]
    y_pool = y[:-n_test]
    
    print(f"  Pool size: {len(X_pool)}, Test size: {n_test}")
    print(f"  Training sizes: {N_SIZES}")
    print(f"  Repeats: {N_REPEATS}")
    
    methods = {
        'MAP': method_map,
        'MCD': method_mcd,
        'Ensemble': method_ensemble,
        'BBB': method_bbb,
        'SWAG': method_swag,
        'CP': method_cp,
    }
    
    results = []
    t_start = time.time()
    
    for n in N_SIZES:
        if n > len(X_pool):
            print(f"  Skipping n={n} (pool too small)")
            continue
        print(f"\n>>> n = {n}")
        
        for rep in range(N_REPEATS):
            idx = np.random.RandomState(rep + n).choice(len(X_pool), n, replace=False)
            X_tr, y_tr = X_pool[idx], y_pool[idx]
            
            for name, func in methods.items():
                t0 = time.time()
                try:
                    if name == 'CP':
                        point_pred, proba, pred_sets = func(X_tr, y_tr, X_te)
                        if proba is None:
                            continue
                        met = compute_metrics(y_te, proba, pred_sets)
                        met['cp_point_accuracy'] = round(float(np.mean(point_pred == y_te)), 4)
                    else:
                        proba, _, _ = func(X_tr, y_tr, X_te)
                        if proba is None:
                            continue
                        met = compute_metrics(y_te, proba)
                    
                    results.append({
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time() - t0, 2),
                        **met
                    })
                except Exception as e:
                    print(f"    ERROR {name}: {e}")
            
            if (rep + 1) % 10 == 0:
                elapsed = (time.time() - t_start) / 60
                print(f"    rep {rep+1}/{N_REPEATS} ({elapsed:.1f} min)")
    
    df = pd.DataFrame(results)
    df.to_csv('uq_results_breast_cancer.csv', index=False)
    
    # Summary
    print("\n" + "=" * 60)
    print("RESULTS SUMMARY")
    print("=" * 60)
    print("\n── BRIER SCORE (lower better) ──")
    print(df.groupby(['method', 'n'])['brier'].mean().unstack().round(4))
    print("\n── NLL (lower better) ──")
    print(df.groupby(['method', 'n'])['nll'].mean().unstack().round(4))
    print("\n── ECE (lower better, target 0) ──")
    print(df.groupby(['method', 'n'])['ece'].mean().unstack().round(4))
    
    if 'cp_coverage' in df.columns:
        print("\n── CP Coverage (target 0.90) ──")
        print(df[df.method == 'CP'].groupby('n')['cp_coverage'].mean().round(4))
    
    return df

# ══════════════════════════════════════════════════════════════
# RUN
# ══════════════════════════════════════════════════════════════
if __name__ == '__main__':
    df = run_experiment()
    print(f"\nResults saved to uq_results_breast_cancer.csv")

In [ ]:
"""
CP Coverage Fix - CP 稳定不了，考虑移除
"""
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import load_breast_cancer
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
HIDDEN = 64
EPOCHS = 500
LR = 1e-3
WEIGHT_DECAY = 1e-4
CAL_FRAC = 0.30
MIN_CAL = 30           # 🔴 新增：最小 calibration 样本数
COVERAGE = 0.90
TEST_FRAC = 0.30

# ══════════════════════════════════════════════════════════════
# MODEL (同上)
# ══════════════════════════════════════════════════════════════
class BinaryMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, 1)
        )
    
    def forward(self, x):
        return self.net(x)
    
    def predict_proba(self, x_np):
        self.eval()
        with torch.no_grad():
            logits = self.forward(torch.tensor(x_np))
        return torch.sigmoid(logits).squeeze().numpy()


def train_adam_with_early_stopping(model, X_tr, y_tr, epochs=EPOCHS, patience=50):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)
    
    Xt = torch.tensor(X_tr)
    yt = torch.tensor(y_tr)
    
    model.train()
    best_loss = float('inf')
    best_state_dict = None
    patience_counter = 0
    
    for epoch in range(epochs):
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(model(Xt).squeeze(), yt)
        loss.backward()
        opt.step()
        scheduler.step(loss)
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter > patience:
                break
    
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return model


def method_cp_fixed(X_tr, y_tr, X_te, verbose=False):
    """
    Split Conformal Prediction - 使用固定最小 calibration set
    """
    # 🔴 关键修改：确保 calibration set 至少 MIN_CAL 个样本
    n_cal = max(MIN_CAL, int(len(X_tr) * CAL_FRAC))
    n_cal = min(n_cal, len(X_tr) // 2)  # 不超过训练集的一半
    
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal], y_tr[:-n_cal]
    
    if verbose:
        print(f"    CP: n_train={len(X_tr)}, n_cal={n_cal}, n_train_cal={len(X_t)}")
    
    model = BinaryMLP(X_tr.shape[1])
    model = train_adam_with_early_stopping(model, X_t, y_t)
    
    proba_c = model.predict_proba(X_c)
    scores = 1 - (proba_c * y_c + (1 - proba_c) * (1 - y_c))
    scores = np.clip(scores, 1e-6, 1 - 1e-6)
    
    alpha = 1 - COVERAGE
    # 标准分位数
    q = np.quantile(scores, np.ceil((n_cal + 1) * (1 - alpha)) / (n_cal + 1))
    
    proba_te = model.predict_proba(X_te)
    pred_set_contains_1 = (1 - proba_te) <= q
    pred_set_contains_0 = proba_te <= q
    
    point_pred = (proba_te >= 0.5).astype(np.float32)
    
    if verbose:
        coverage = np.mean(
            ((y_te == 0) & pred_set_contains_0) | ((y_te == 1) & pred_set_contains_1)
        )
        print(f"    CP: q={q:.4f}, coverage={coverage:.4f}")
    
    return point_pred, proba_te, (pred_set_contains_0, pred_set_contains_1)


def quick_test():
    print("=" * 70)
    print("CP Coverage Test (MIN_CAL=30, CAL_FRAC=0.30)")
    print("=" * 70)
    
    X, y = load_breast_cancer(return_X_y=True)
    X = X.astype(np.float32)
    y = y.astype(np.float32)
    
    sc = StandardScaler()
    X = sc.fit_transform(X)
    
    n_test = int(len(X) * TEST_FRAC)
    X_te = X[-n_test:]
    y_te = y[-n_test:]
    X_pool = X[:-n_test]
    y_pool = y[:-n_test]
    
    print(f"Pool size: {len(X_pool)}, Test size: {n_test}")
    print(f"MIN_CAL = {MIN_CAL}, CAL_FRAC = {CAL_FRAC}")
    print("-" * 70)
    
    n_sizes = [30, 50, 100, 200]
    n_repeats = 10
    
    for n in n_sizes:
        if n > len(X_pool):
            print(f"  n={n} skipped")
            continue
        
        coverages = []
        
        for rep in range(n_repeats):
            idx = np.random.RandomState(rep + n).choice(len(X_pool), n, replace=False)
            X_tr, y_tr = X_pool[idx], y_pool[idx]
            
            _, proba, (set_0, set_1) = method_cp_fixed(X_tr, y_tr, X_te, verbose=False)
            
            coverage = np.mean(
                ((y_te == 0) & set_0) | ((y_te == 1) & set_1)
            )
            coverages.append(coverage)
        
        print(f"\n  n={n}:")
        print(f"    mean coverage = {np.mean(coverages):.4f} ± {np.std(coverages):.4f}")
        print(f"    min = {np.min(coverages):.4f}, max = {np.max(coverages):.4f}")
        print(f"    target = {COVERAGE}")
        
        if np.mean(coverages) < 0.85:
            print(f"    ⚠️  Below target")
        elif np.mean(coverages) > 0.95:
            print(f"    ⚠️  Above target (conservative)")
        else:
            print(f"    ✅ Acceptable")
        
        print("-" * 70)


if __name__ == '__main__':
    quick_test()

In [ ]:
"""
UQ Experiment - Classification (UCI Heart Disease)
===================================================
Dataset: Cleveland Heart Disease (303 samples, 13 features)
Binary classification: presence (1) or absence (0) of heart disease

Methods: MAP, MCD, Ensemble, BBB, SWAG, CP
Metrics: Brier Score, NLL, ECE
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ══════════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════════
HIDDEN = 64
EPOCHS = 500
LR = 1e-3
WEIGHT_DECAY = 1e-4
DROPOUT_P = 0.1
N_ENSEMBLE = 5
MC_SAMPLES = 50
SWAG_SAMPLES = 30
SWAG_START = 0.60
BBB_KL_W = 1e-4
COVERAGE = 0.90
CAL_FRAC = 0.30
TEST_FRAC = 0.30

# Heart Disease specific
N_SIZES = [30, 50, 100, 150]  # 最大不超过 pool 的 80%
N_REPEATS = 50

def load_heart_disease():
    """
    UCI Heart Disease (Cleveland) dataset.
    303 samples, 13 features, binary classification.
    """
    url = "https://archive.ics.uci.edu/ml/machine-learning-databases/heart-disease/processed.cleveland.data"

    column_names = [
        'age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg',
        'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal', 'target'
    ]
    
    # 将 '?' 标记为 NaN
    df = pd.read_csv(url, header=None, names=column_names, na_values='?')
    
    # 分离特征和标签
    X = df.drop('target', axis=1).values.astype(np.float32)
    y = df['target'].values
    
    # 将目标变量二值化（0 = 无心脏病，1 = 有心脏病）
    # 原始数据中 0 = 无，1,2,3,4 = 不同程度的心脏病
    y = (y > 0).astype(np.float32)
    
    # 处理缺失值：用每列的中位数填充
    # 注意：ca 和 thal 列有缺失值
    imputer = SimpleImputer(strategy='median')
    X = imputer.fit_transform(X)
    
    print(f"  Heart Disease: X={X.shape}, y={y.shape}")
    print(f"  Class balance: {y.mean():.3f} (positive proportion)")
    print(f"  Missing values handled: median imputation")
    
    return X.astype(np.float32), y.astype(np.float32)

# ══════════════════════════════════════════════════════════════
# MODEL (same as Breast Cancer)
# ══════════════════════════════════════════════════════════════
class BinaryMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 1)
        )
    
    def forward(self, x):
        return self.net(x)
    
    def predict_proba(self, x_np):
        self.eval()
        with torch.no_grad():
            logits = self.forward(torch.tensor(x_np))
        return torch.sigmoid(logits).squeeze().numpy()


class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        w = self.wm + ws * torch.randn_like(self.wm)
        b = self.bm + bs * torch.randn_like(self.bm)
        return nn.functional.linear(x, w, b)
    
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)


class BayesBinaryMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1 = BayesLinear(in_dim, hidden)
        self.l2 = BayesLinear(hidden, hidden)
        self.lo = BayesLinear(hidden, 1)
        self.act = nn.ReLU()
    
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    
    def kl(self):
        return self.l1.kl() + self.l2.kl() + self.lo.kl()


# ══════════════════════════════════════════════════════════════
# TRAINING
# ══════════════════════════════════════════════════════════════
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(model(Xt).squeeze(), yt)
        loss.backward()
        opt.step()
    return model


def train_adam_with_early_stopping(model, X_tr, y_tr, epochs=EPOCHS, patience=50):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(opt, patience=20, factor=0.5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    best_loss = float('inf')
    best_state_dict = None
    patience_counter = 0
    
    for epoch in range(epochs):
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(model(Xt).squeeze(), yt)
        loss.backward()
        opt.step()
        scheduler.step(loss)
        
        if loss.item() < best_loss:
            best_loss = loss.item()
            best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter > patience:
                break
    
    if best_state_dict is not None:
        model.load_state_dict(best_state_dict)
    return model


# ══════════════════════════════════════════════════════════════
# METHODS
# ══════════════════════════════════════════════════════════════
def method_map(X_tr, y_tr, X_te):
    model = BinaryMLP(X_tr.shape[1])
    model = train_adam(model, X_tr, y_tr)
    proba = model.predict_proba(X_te)
    return proba, None, None


def method_mcd(X_tr, y_tr, X_te):
    model = BinaryMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    model = train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        logits = torch.stack([model(torch.tensor(X_te)) for _ in range(MC_SAMPLES)])
    proba = torch.sigmoid(logits).squeeze().numpy()
    return proba.mean(axis=0), None, None


def method_ensemble(X_tr, y_tr, X_te):
    probas = []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = BinaryMLP(X_tr.shape[1])
        model = train_adam(model, X_tr, y_tr)
        probas.append(model.predict_proba(X_te))
    probas = np.array(probas)
    return probas.mean(axis=0), None, None


def method_bbb(X_tr, y_tr, X_te):
    model = BayesBinaryMLP(X_tr.shape[1])
    opt = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(model(Xt).squeeze(), yt) + BBB_KL_W * model.kl()
        loss.backward()
        opt.step()
    model.train()
    with torch.no_grad():
        logits = torch.stack([model(torch.tensor(X_te)) for _ in range(MC_SAMPLES)])
    proba = torch.sigmoid(logits).squeeze().numpy()
    return proba.mean(axis=0), None, None


def method_swag(X_tr, y_tr, X_te):
    model = BinaryMLP(X_tr.shape[1])
    opt = optim.SGD(model.parameters(), lr=LR * 10, momentum=0.9, weight_decay=WEIGHT_DECAY)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []
    start = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = nn.functional.binary_cross_entropy_with_logits(model(Xt).squeeze(), yt)
        if torch.isnan(loss):
            break
        loss.backward()
        opt.step()
        scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten() for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    
    if len(w_list) < 2:
        return None, None, None
    
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    
    probas = []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape))
            ptr += n
        probas.append(model.predict_proba(X_te))
    probas = np.array(probas)
    return probas.mean(axis=0), None, None


def method_cp(X_tr, y_tr, X_te):
    """Split Conformal Prediction"""
    n_cal = max(25, int(len(X_tr) * CAL_FRAC))
    n_cal = min(n_cal, len(X_tr) // 2)
    
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal], y_tr[:-n_cal]
    
    model = BinaryMLP(X_tr.shape[1])
    model = train_adam_with_early_stopping(model, X_t, y_t)
    
    proba_c = model.predict_proba(X_c)
    scores = 1 - (proba_c * y_c + (1 - proba_c) * (1 - y_c))
    scores = np.clip(scores, 1e-6, 1 - 1e-6)
    
    alpha = 1 - COVERAGE
    q = np.quantile(scores, np.ceil((n_cal + 1) * (1 - alpha)) / (n_cal + 1))
    
    proba_te = model.predict_proba(X_te)
    pred_set_contains_1 = (1 - proba_te) <= q
    pred_set_contains_0 = proba_te <= q
    
    point_pred = (proba_te >= 0.5).astype(np.float32)
    
    return point_pred, proba_te, (pred_set_contains_0, pred_set_contains_1)


# ══════════════════════════════════════════════════════════════
# METRICS
# ══════════════════════════════════════════════════════════════
def compute_brier_score(y_true, proba):
    return float(np.mean((proba - y_true) ** 2))


def compute_ece(y_true, proba, n_bins=10):
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0
    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]
        in_bin = (proba >= bin_lower) & (proba < bin_upper)
        if bin_upper == 1.0:
            in_bin = (proba >= bin_lower) & (proba <= bin_upper)
        if not np.any(in_bin):
            continue
        prop_in_bin = np.mean(in_bin)
        acc_in_bin = np.mean(y_true[in_bin])
        conf_in_bin = np.mean(proba[in_bin])
        ece += prop_in_bin * np.abs(acc_in_bin - conf_in_bin)
    return float(ece)


def compute_metrics(y_true, proba, pred_set_info=None):
    brier = compute_brier_score(y_true, proba)
    nll = float(-np.mean(y_true * np.log(proba + 1e-10) + 
                         (1 - y_true) * np.log(1 - proba + 1e-10)))
    ece = compute_ece(y_true, proba)
    
    metrics = {
        'brier': round(brier, 4),
        'nll': round(nll, 4),
        'ece': round(ece, 4),
    }
    
    if pred_set_info is not None:
        contains_0, contains_1 = pred_set_info
        coverage = np.mean(((y_true == 0) & contains_0) | ((y_true == 1) & contains_1))
        set_size = np.mean(contains_0.astype(int) + contains_1.astype(int))
        metrics['cp_coverage'] = round(coverage, 4)
        metrics['cp_set_size'] = round(set_size, 4)
    
    return metrics


# ══════════════════════════════════════════════════════════════
# MAIN EXPERIMENT
# ══════════════════════════════════════════════════════════════
def run_experiment():
    print("=" * 60)
    print("Classification UQ Benchmark - UCI Heart Disease")
    print("=" * 60)
    
    X, y = load_heart_disease()
    
    # Standardize
    sc_X = StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    
    # Fixed test split
    n_test = int(len(X) * TEST_FRAC)
    X_te = X[-n_test:]
    y_te = y[-n_test:]
    X_pool = X[:-n_test]
    y_pool = y[:-n_test]
    
    # Adjust n_sizes based on pool
    max_n = int(len(X_pool) * 0.8)
    n_sizes = [n for n in N_SIZES if n <= max_n]
    
    print(f"  Pool size: {len(X_pool)}, Test size: {n_test}")
    print(f"  Training sizes: {n_sizes}")
    print(f"  Repeats: {N_REPEATS}")
    
    methods = {
        'MAP': method_map,
        'MCD': method_mcd,
        'Ensemble': method_ensemble,
        'BBB': method_bbb,
        'SWAG': method_swag,
        'CP': method_cp,
    }
    
    results = []
    t_start = time.time()
    
    for n in n_sizes:
        print(f"\n>>> n = {n}")
        
        for rep in range(N_REPEATS):
            idx = np.random.RandomState(rep + n).choice(len(X_pool), n, replace=False)
            X_tr, y_tr = X_pool[idx], y_pool[idx]
            
            for name, func in methods.items():
                t0 = time.time()
                try:
                    if name == 'CP':
                        point_pred, proba, pred_sets = func(X_tr, y_tr, X_te)
                        if proba is None:
                            continue
                        met = compute_metrics(y_te, proba, pred_sets)
                    else:
                        proba, _, _ = func(X_tr, y_tr, X_te)
                        if proba is None:
                            continue
                        met = compute_metrics(y_te, proba)
                    
                    results.append({
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time() - t0, 2),
                        **met
                    })
                except Exception as e:
                    print(f"    ERROR {name}: {e}")
            
            if (rep + 1) % 10 == 0:
                elapsed = (time.time() - t_start) / 60
                print(f"    rep {rep+1}/{N_REPEATS} ({elapsed:.1f} min)")
    
    df = pd.DataFrame(results)
    df.to_csv('uq_results_heart_disease.csv', index=False)
    
    # Summary
    print("\n" + "=" * 60)
    print("RESULTS SUMMARY")
    print("=" * 60)
    print("\n── BRIER SCORE (lower better) ──")
    print(df.groupby(['method', 'n'])['brier'].mean().unstack().round(4))
    print("\n── NLL (lower better) ──")
    print(df.groupby(['method', 'n'])['nll'].mean().unstack().round(4))
    print("\n── ECE (lower better) ──")
    print(df.groupby(['method', 'n'])['ece'].mean().unstack().round(4))
    
    if 'cp_coverage' in df.columns:
        print("\n── CP Coverage (target 0.90) ──")
        cp_df = df[df.method == 'CP'].groupby('n')['cp_coverage'].agg(['mean', 'std'])
        print(cp_df.round(4))
    
    print(f"\nResults saved to uq_results_heart_disease.csv")
    return df


if __name__ == '__main__':
    df = run_experiment()

In [ ]:
"""
UQ Experiment — Additional Real Datasets
=========================================
New datasets: Yacht, Boston Housing, Kin8nm, Naval Propulsion
Same pipeline as uq_real_datasets.py
Each dataset saves immediately after completion.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
import properscoring as ps
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import time
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ── Config (identical to original) ───────────────────────────
N_REPEATS    = 50
EPOCHS       = 500
LR           = 1e-3
LR_SGD       = 0.01
HIDDEN       = 64
DROPOUT_P    = 0.1
N_ENSEMBLE   = 5
MC_SAMPLES   = 50
SWAG_SAMPLES = 30
SWAG_START   = 0.60
BBB_KL_W     = 1e-4
COVERAGE     = 0.90
CAL_FRAC     = 0.20
TEST_FRAC    = 0.30
SIGMA_FLOOR  = 0.05
N_SIZES_CANDIDATES = [30, 50, 100, 200, 500]

# ── Data loading ──────────────────────────────────────────────
def load_yacht():
    """
    UCI Yacht Hydrodynamics.
    308 samples, 6 features, target: residuary resistance.
    """
    try:
        data = fetch_openml(name='yacht_hydrodynamics', version=1,
                            as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        # Direct download fallback
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "00243/yacht_hydrodynamics.data")
        df = pd.read_csv(url, sep=r'\s+', header=None)
        X = df.iloc[:, :6].values.astype(np.float32)
        y = df.iloc[:, 6].values.astype(np.float32)
    print(f"  Yacht: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.2f}, {y.max():.2f}]")
    return X, y

def load_boston():
    """
    Boston Housing dataset.
    506 samples, 13 features, target: median house value.
    Note: fetched via OpenML (dataset 531) to avoid sklearn deprecation.
    """
    try:
        data = fetch_openml(data_id=531, as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        # Fallback: load from alternative source
        url = ("https://raw.githubusercontent.com/selva86/datasets/"
               "master/BostonHousing.csv")
        df = pd.read_csv(url)
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values.astype(np.float32)
    print(f"  Boston: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.1f}, {y.max():.1f}]")
    return X, y

def load_kin8nm():
    """
    Kin8nm dataset (robot arm kinematics).
    8192 samples, 8 features, target: distance.
    """
    try:
        data = fetch_openml(name='kin8nm', version=1,
                            as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        url = ("https://raw.githubusercontent.com/luishpmendes/datasets/"
               "master/kin8nm/kin8nm.csv")
        df = pd.read_csv(url)
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values.astype(np.float32)
    print(f"  Kin8nm: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.3f}, {y.max():.3f}]")
    return X, y

def load_naval():
    """
    Naval Propulsion Plant dataset.
    11934 samples, 16 features, target: turbine decay (column 17).
    """
    try:
        data = fetch_openml(name='naval-propulsion-plant', version=1,
                            as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "00316/UCI%20CBM%20Dataset.zip")
        # If zip fails, use a known mirror
        import io, zipfile, requests
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        fname = [f for f in z.namelist() if f.endswith('.txt')][0]
        df = pd.read_csv(z.open(fname), sep=r'\s+', header=None)
        X = df.iloc[:, :16].values.astype(np.float32)
        y = df.iloc[:, 16].values.astype(np.float32)  # turbine decay
    print(f"  Naval: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.4f}, {y.max():.4f}]")
    return X, y

# ── Loss ──────────────────────────────────────────────────────
def gaussian_nll(out, y_t):
    mu, lv = out[:, 0], out[:, 1]
    var = torch.exp(lv).clamp(1e-3, 1e3)
    return (0.5 * ((y_t - mu)**2 / var + lv)).mean()

# ── Models (identical to original) ───────────────────────────
class HeteroscedasticMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 2))
    def forward(self, x): return self.net(x)
    def predict_gaussian(self, x_np):
        self.eval()
        with torch.no_grad():
            out = self.forward(torch.tensor(x_np))
        mu    = out[:, 0].numpy()
        sigma = np.exp(0.5 * out[:, 1].numpy()).clip(1e-4, 1e2)
        return mu, sigma

class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        return nn.functional.linear(
            x, self.wm + ws * torch.randn_like(self.wm),
               self.bm + bs * torch.randn_like(self.bm))
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1  = BayesLinear(in_dim, hidden)
        self.l2  = BayesLinear(hidden, hidden)
        self.lo  = BayesLinear(hidden, 2)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    def kl(self): return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ── Training ──────────────────────────────────────────────────
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); gaussian_nll(model(Xt), yt).backward(); opt.step()

# ── Methods (identical to original) ──────────────────────────
def method_map(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_tr, y_tr)
    return model.predict_gaussian(X_te), None, None

def method_mcd(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te))
                            for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) +
                       torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_ensemble(X_tr, y_tr, X_te):
    mus, sigsqs = [], []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = HeteroscedasticMLP(X_tr.shape[1])
        train_adam(model, X_tr, y_tr)
        m, s = model.predict_gaussian(X_te)
        mus.append(m); sigsqs.append(s**2)
    mus, sigsqs = np.array(mus), np.array(sigsqs)
    return (mus.mean(0), np.sqrt(mus.var(0) + sigsqs.mean(0))), None, None

def method_bbb(X_tr, y_tr, X_te):
    model = BayesMLP(X_tr.shape[1])
    opt   = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt) + BBB_KL_W * model.kl()
        loss.backward(); opt.step()
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te))
                            for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) +
                       torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_swag(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    opt   = optim.SGD(model.parameters(), lr=LR_SGD,
                      momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=LR_SGD * 0.01)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []; start = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt)
        if torch.isnan(loss): break
        loss.backward(); opt.step(); scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten()
                                for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    if len(w_list) < 2:
        return None, None, None
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    mus, sigsqs = [], []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape)); ptr += n
        mu, sig = model.predict_gaussian(X_te)
        sig = np.clip(sig, 1e-4, 5.0)
        mus.append(mu); sigsqs.append(sig**2)
    mus = np.array(mus)
    return (mus.mean(0), np.sqrt(mus.var(0) +
                                  np.array(sigsqs).mean(0))), None, None

def method_cp(X_tr, y_tr, X_te):
    n_cal = max(15, int(len(X_tr) * CAL_FRAC))
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal],  y_tr[:-n_cal]
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_t, y_t)
    mu_c, _ = model.predict_gaussian(X_c)
    alpha   = 1 - COVERAGE
    q       = np.quantile(np.abs(y_c - mu_c),
                          min(1.0, np.ceil((n_cal+1)*(1-alpha))/n_cal))
    mu_te, _ = model.predict_gaussian(X_te)
    z         = stats.norm.ppf(1 - alpha/2)
    return (mu_te, np.full(len(mu_te), q/z)), mu_te - q, mu_te + q

# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(y_true, mu, sigma, lower=None, upper=None):
    sigma_clipped = np.clip(sigma, SIGMA_FLOOR, None)
    crps    = float(ps.crps_gaussian(y_true, mu, sigma_clipped).mean())
    nll     = float(0.5 * np.mean(np.log(2*np.pi*sigma_clipped**2)
                                  + (y_true-mu)**2/sigma_clipped**2))
    sigma_raw = np.clip(sigma, 1e-4, None)
    nll_raw   = float(0.5 * np.mean(np.log(2*np.pi*sigma_raw**2)
                                    + (y_true-mu)**2/sigma_raw**2))
    alpha = 1 - COVERAGE
    z     = stats.norm.ppf(1 - alpha/2)
    lo    = lower if lower is not None else mu - z * sigma
    hi    = upper if upper is not None else mu + z * sigma
    picp  = float(np.mean((y_true >= lo) & (y_true <= hi)))
    mpiw  = float(np.mean(hi - lo))
    pen_lo   = (2/alpha) * (lo - y_true) * (y_true < lo)
    pen_hi   = (2/alpha) * (y_true - hi) * (y_true > hi)
    is_score = float(np.mean((hi - lo) + pen_lo + pen_hi))
    return {
        'crps':           round(crps, 4),
        'nll':            round(nll, 4),
        'nll_raw':        round(nll_raw, 4),
        'picp':           round(picp, 4),
        'mpiw':           round(mpiw, 4),
        'interval_score': round(is_score, 4),
        'sigma_mean':     round(float(sigma.mean()), 4),
    }

# ── Main loop ─────────────────────────────────────────────────
def run_dataset(dataset_name, X, y):
    print(f"\n{'='*55}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*55}")

    sc_X, sc_y = StandardScaler(), StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    y = sc_y.fit_transform(y.reshape(-1,1)).ravel().astype(np.float32)

    n_total = len(X)
    n_test  = int(n_total * TEST_FRAC)
    X_te, y_te     = X[-n_test:], y[-n_test:]
    X_pool, y_pool = X[:-n_test],  y[:-n_test]

    max_n   = int(len(X_pool) * 0.8)
    n_sizes = [n for n in N_SIZES_CANDIDATES if n <= max_n]
    print(f"  Total={n_total}, pool={len(X_pool)}, "
          f"test={n_test}, n_sizes={n_sizes}")

    methods = {
        'MAP':      method_map,
        'MCD':      method_mcd,
        'Ensemble': method_ensemble,
        'BBB':      method_bbb,
        'SWAG':     method_swag,
        'CP':       method_cp,
    }

    output_file = f"uq_results_{dataset_name}.csv"
    ckpt_file   = f"uq_checkpoint_{dataset_name}.csv"
    results     = []
    t_start     = time.time()

    for n in n_sizes:
        print(f"\n>>> n = {n}", flush=True)
        for rep in range(N_REPEATS):
            idx = np.random.RandomState(rep + n).choice(
                len(X_pool), n, replace=False)
            X_tr, y_tr = X_pool[idx], y_pool[idx]

            for name, func in methods.items():
                t0 = time.time()
                try:
                    result = func(X_tr, y_tr, X_te)
                    if result[0] is None:
                        results.append({
                            'dataset': dataset_name,
                            'method': name, 'n': n, 'rep': rep,
                            'time_s': round(time.time()-t0, 2),
                            'swag_ok': False,
                            'crps': np.nan, 'nll': np.nan,
                            'nll_raw': np.nan, 'picp': np.nan,
                            'mpiw': np.nan, 'interval_score': np.nan,
                            'sigma_mean': np.nan})
                        continue
                    (mu, sig), lo, hi = result
                    met = compute_metrics(y_te, mu, sig,
                                         lower=lo, upper=hi)
                    results.append({
                        'dataset': dataset_name,
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time()-t0, 2),
                        'swag_ok': True, **met})
                except Exception as e:
                    print(f"  ERROR {name} n={n} rep={rep}: {e}")

            if (rep+1) % 10 == 0:
                elapsed = (time.time()-t_start)/60
                print(f"  rep {rep+1}/{N_REPEATS}  ({elapsed:.1f} min)",
                      flush=True)
                pd.DataFrame(results).to_csv(ckpt_file, index=False)

    df = pd.DataFrame(results)
    # ── Save immediately after each dataset ──────────────────
    df.to_csv(output_file, index=False)
    print(f"\n✓ Saved → {output_file}")

    # Quick summary
    print(f"\n── CRPS [{dataset_name}]")
    grp = df.groupby(['method','n'])['crps']
    mean_df = grp.mean().unstack().round(3)
    std_df  = grp.std().unstack().round(3)
    for col in mean_df.columns:
        mean_df[col] = (mean_df[col].astype(str) + " ±"
                        + std_df[col].astype(str))
    print(mean_df.to_string())

    print(f"\n── PICP (target 0.90) [{dataset_name}]")
    print(df.groupby(['method','n'])['picp'].mean().unstack().round(3)
            .to_string())

    sw = df[df.method=='SWAG'].groupby('n')['swag_ok'].agg(['sum','count'])
    sw['rate'] = (sw['sum']/sw['count']).round(2)
    print(f"\n── SWAG convergence [{dataset_name}]")
    print(sw.to_string())

    return df

# ── Run all datasets ──────────────────────────────────────────
if __name__ == '__main__':
    t0_total = time.time()
    all_dfs  = []

    datasets_to_run = {
        'yacht':   load_yacht,
        'boston':  load_boston,
        'kin8nm':  load_kin8nm,
        'naval':   load_naval,
    }

    print("Loading datasets...")
    for name, loader in datasets_to_run.items():
        try:
            X, y = loader()
            df = run_dataset(name, X, y)
            all_dfs.append(df)
        except Exception as e:
            print(f"\n  ✗ {name} failed: {e}")
            continue

    # Merge with existing real dataset results if present
    existing_files = [
        'uq_results_energy.csv',
        'uq_results_concrete.csv',
    ]
    for f in existing_files:
        try:
            all_dfs.append(pd.read_csv(f))
            print(f"  Loaded existing: {f}")
        except FileNotFoundError:
            pass

    if all_dfs:
        combined = pd.concat(all_dfs, ignore_index=True)
        combined.to_csv('uq_results_all_datasets.csv', index=False)
        print(f"\n✓ All datasets combined → uq_results_all_datasets.csv")
        print(f"  Datasets: {combined['dataset'].unique()}")
        print(f"  Total rows: {len(combined)}")

    print(f"\nTotal runtime: {(time.time()-t0_total)/60:.1f} min")

In [ ]:
"""
UQ Experiment — Kin8nm + Naval only
跑完每个dataset立刻保存，同时保存checkpoint防止丢失
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from scipy import stats
import properscoring as ps
from sklearn.preprocessing import StandardScaler
from sklearn.datasets import fetch_openml
import time, os
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ── Config ────────────────────────────────────────────────────
N_REPEATS    = 50
EPOCHS       = 500
LR           = 1e-3
LR_SGD       = 0.01
HIDDEN       = 64
DROPOUT_P    = 0.1
N_ENSEMBLE   = 5
MC_SAMPLES   = 50
SWAG_SAMPLES = 30
SWAG_START   = 0.60
BBB_KL_W     = 1e-4
COVERAGE     = 0.90
CAL_FRAC     = 0.20
TEST_FRAC    = 0.30
SIGMA_FLOOR  = 0.05
N_SIZES_CANDIDATES = [30, 50, 100, 200, 500]

# ── Data loading ──────────────────────────────────────────────
def load_kin8nm():
    try:
        data = fetch_openml(name='kin8nm', version=1,
                            as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        url = ("https://raw.githubusercontent.com/luishpmendes/datasets/"
               "master/kin8nm/kin8nm.csv")
        df = pd.read_csv(url)
        X = df.iloc[:, :-1].values.astype(np.float32)
        y = df.iloc[:, -1].values.astype(np.float32)
    print(f"  Kin8nm: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.3f}, {y.max():.3f}]")
    return X, y

def load_naval():
    try:
        data = fetch_openml(name='naval-propulsion-plant', version=1,
                            as_frame=True, parser='auto')
        X = data.data.values.astype(np.float32)
        y = data.target.values.astype(np.float32)
    except Exception:
        import io, zipfile, requests
        url = ("https://archive.ics.uci.edu/ml/machine-learning-databases/"
               "00316/UCI%20CBM%20Dataset.zip")
        r = requests.get(url)
        z = zipfile.ZipFile(io.BytesIO(r.content))
        fname = [f for f in z.namelist() if f.endswith('.txt')][0]
        df = pd.read_csv(z.open(fname), sep=r'\s+', header=None)
        X = df.iloc[:, :16].values.astype(np.float32)
        y = df.iloc[:, 16].values.astype(np.float32)
    print(f"  Naval: X={X.shape}, y={y.shape}, "
          f"y_range=[{y.min():.4f}, {y.max():.4f}]")
    return X, y

# ── Loss ──────────────────────────────────────────────────────
def gaussian_nll(out, y_t):
    mu, lv = out[:, 0], out[:, 1]
    var = torch.exp(lv).clamp(1e-3, 1e3)
    return (0.5 * ((y_t - mu)**2 / var + lv)).mean()

# ── Models ────────────────────────────────────────────────────
class HeteroscedasticMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN, dropout_p=0.0):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Dropout(dropout_p),
            nn.Linear(hidden, 2))
    def forward(self, x): return self.net(x)
    def predict_gaussian(self, x_np):
        self.eval()
        with torch.no_grad():
            out = self.forward(torch.tensor(x_np))
        mu    = out[:, 0].numpy()
        sigma = np.exp(0.5 * out[:, 1].numpy()).clip(1e-4, 1e2)
        return mu, sigma

class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        return nn.functional.linear(
            x, self.wm + ws * torch.randn_like(self.wm),
               self.bm + bs * torch.randn_like(self.bm))
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std) - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesMLP(nn.Module):
    def __init__(self, in_dim, hidden=HIDDEN):
        super().__init__()
        self.l1  = BayesLinear(in_dim, hidden)
        self.l2  = BayesLinear(hidden, hidden)
        self.lo  = BayesLinear(hidden, 2)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    def kl(self): return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ── Training ──────────────────────────────────────────────────
def train_adam(model, X_tr, y_tr, epochs=EPOCHS):
    opt = optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(epochs):
        opt.zero_grad(); gaussian_nll(model(Xt), yt).backward(); opt.step()

# ── Methods ───────────────────────────────────────────────────
def method_map(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_tr, y_tr)
    return model.predict_gaussian(X_te), None, None

def method_mcd(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1], dropout_p=DROPOUT_P)
    train_adam(model, X_tr, y_tr)
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te))
                            for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) +
                       torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_ensemble(X_tr, y_tr, X_te):
    mus, sigsqs = [], []
    for k in range(N_ENSEMBLE):
        torch.manual_seed(k + 42)
        model = HeteroscedasticMLP(X_tr.shape[1])
        train_adam(model, X_tr, y_tr)
        m, s = model.predict_gaussian(X_te)
        mus.append(m); sigsqs.append(s**2)
    mus, sigsqs = np.array(mus), np.array(sigsqs)
    return (mus.mean(0), np.sqrt(mus.var(0) + sigsqs.mean(0))), None, None

def method_bbb(X_tr, y_tr, X_te):
    model = BayesMLP(X_tr.shape[1])
    opt   = optim.Adam(model.parameters(), lr=LR)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    model.train()
    for _ in range(EPOCHS * 2):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt) + BBB_KL_W * model.kl()
        loss.backward(); opt.step()
    model.train()
    with torch.no_grad():
        outs = torch.stack([model(torch.tensor(X_te))
                            for _ in range(MC_SAMPLES)])
    mu    = outs[..., 0].mean(0).numpy()
    sigma = torch.sqrt(outs[..., 0].var(0) +
                       torch.exp(outs[..., 1]).mean(0)).numpy()
    return (mu, sigma), None, None

def method_swag(X_tr, y_tr, X_te):
    model = HeteroscedasticMLP(X_tr.shape[1])
    opt   = optim.SGD(model.parameters(), lr=LR_SGD,
                      momentum=0.9, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        opt, T_max=EPOCHS, eta_min=LR_SGD * 0.01)
    Xt, yt = torch.tensor(X_tr), torch.tensor(y_tr)
    w_list = []; start = int(EPOCHS * SWAG_START)
    model.train()
    for ep in range(EPOCHS):
        opt.zero_grad()
        loss = gaussian_nll(model(Xt), yt)
        if torch.isnan(loss): break
        loss.backward(); opt.step(); scheduler.step()
        if ep >= start and ep % 5 == 0:
            w_snap = torch.cat([p.detach().flatten()
                                for p in model.parameters()])
            if not torch.isnan(w_snap).any():
                w_list.append(w_snap)
    if len(w_list) < 2:
        return None, None, None
    W = torch.stack(w_list)
    w_mean, w_var = W.mean(0), W.var(0).clamp(1e-8)
    if torch.isnan(w_mean).any():
        return None, None, None
    mus, sigsqs = [], []
    for _ in range(SWAG_SAMPLES):
        w_s = w_mean + torch.randn_like(w_mean) * w_var.sqrt()
        ptr = 0
        for p in model.parameters():
            n = p.numel()
            p.data.copy_(w_s[ptr:ptr+n].view(p.shape)); ptr += n
        mu, sig = model.predict_gaussian(X_te)
        sig = np.clip(sig, 1e-4, 5.0)
        mus.append(mu); sigsqs.append(sig**2)
    mus = np.array(mus)
    return (mus.mean(0), np.sqrt(mus.var(0) +
                                  np.array(sigsqs).mean(0))), None, None

def method_cp(X_tr, y_tr, X_te):
    n_cal = max(15, int(len(X_tr) * CAL_FRAC))
    X_c, y_c = X_tr[-n_cal:], y_tr[-n_cal:]
    X_t, y_t = X_tr[:-n_cal],  y_tr[:-n_cal]
    model = HeteroscedasticMLP(X_tr.shape[1])
    train_adam(model, X_t, y_t)
    mu_c, _ = model.predict_gaussian(X_c)
    alpha   = 1 - COVERAGE
    q       = np.quantile(np.abs(y_c - mu_c),
                          min(1.0, np.ceil((n_cal+1)*(1-alpha))/n_cal))
    mu_te, _ = model.predict_gaussian(X_te)
    z         = stats.norm.ppf(1 - alpha/2)
    return (mu_te, np.full(len(mu_te), q/z)), mu_te - q, mu_te + q

# ── Metrics ───────────────────────────────────────────────────
def compute_metrics(y_true, mu, sigma, lower=None, upper=None):
    sigma_clipped = np.clip(sigma, SIGMA_FLOOR, None)
    crps    = float(ps.crps_gaussian(y_true, mu, sigma_clipped).mean())
    nll     = float(0.5 * np.mean(np.log(2*np.pi*sigma_clipped**2)
                                  + (y_true-mu)**2/sigma_clipped**2))
    sigma_raw = np.clip(sigma, 1e-4, None)
    nll_raw   = float(0.5 * np.mean(np.log(2*np.pi*sigma_raw**2)
                                    + (y_true-mu)**2/sigma_raw**2))
    alpha = 1 - COVERAGE
    z     = stats.norm.ppf(1 - alpha/2)
    lo    = lower if lower is not None else mu - z * sigma
    hi    = upper if upper is not None else mu + z * sigma
    picp  = float(np.mean((y_true >= lo) & (y_true <= hi)))
    mpiw  = float(np.mean(hi - lo))
    pen_lo   = (2/alpha) * (lo - y_true) * (y_true < lo)
    pen_hi   = (2/alpha) * (y_true - hi) * (y_true > hi)
    is_score = float(np.mean((hi - lo) + pen_lo + pen_hi))
    return {
        'crps':           round(crps, 4),
        'nll':            round(nll, 4),
        'nll_raw':        round(nll_raw, 4),
        'picp':           round(picp, 4),
        'mpiw':           round(mpiw, 4),
        'interval_score': round(is_score, 4),
        'sigma_mean':     round(float(sigma.mean()), 4),
    }

# ── Main loop ─────────────────────────────────────────────────
def run_dataset(dataset_name, X, y):
    print(f"\n{'='*55}")
    print(f"Dataset: {dataset_name.upper()}")
    print(f"{'='*55}")

    sc_X, sc_y = StandardScaler(), StandardScaler()
    X = sc_X.fit_transform(X).astype(np.float32)
    y = sc_y.fit_transform(y.reshape(-1,1)).ravel().astype(np.float32)

    n_total = len(X)
    n_test  = int(n_total * TEST_FRAC)
    X_te, y_te     = X[-n_test:], y[-n_test:]
    X_pool, y_pool = X[:-n_test],  y[:-n_test]

    max_n   = int(len(X_pool) * 0.8)
    n_sizes = [n for n in N_SIZES_CANDIDATES if n <= max_n]
    print(f"  Total={n_total}, pool={len(X_pool)}, "
          f"test={n_test}, n_sizes={n_sizes}")

    methods = {
        'MAP':      method_map,
        'MCD':      method_mcd,
        'Ensemble': method_ensemble,
        'BBB':      method_bbb,
        'SWAG':     method_swag,
        'CP':       method_cp,
    }

    output_file = f"uq_results_{dataset_name}.csv"
    ckpt_file   = f"uq_checkpoint_{dataset_name}.csv"
    results     = []
    t_start     = time.time()

    for n in n_sizes:
        print(f"\n>>> n = {n}", flush=True)
        for rep in range(N_REPEATS):
            idx = np.random.RandomState(rep + n).choice(
                len(X_pool), n, replace=False)
            X_tr, y_tr = X_pool[idx], y_pool[idx]

            for name, func in methods.items():
                t0 = time.time()
                try:
                    result = func(X_tr, y_tr, X_te)
                    if result[0] is None:
                        results.append({
                            'dataset': dataset_name,
                            'method': name, 'n': n, 'rep': rep,
                            'time_s': round(time.time()-t0, 2),
                            'swag_ok': False,
                            'crps': np.nan, 'nll': np.nan,
                            'nll_raw': np.nan, 'picp': np.nan,
                            'mpiw': np.nan, 'interval_score': np.nan,
                            'sigma_mean': np.nan})
                        continue
                    (mu, sig), lo, hi = result
                    met = compute_metrics(y_te, mu, sig,
                                         lower=lo, upper=hi)
                    results.append({
                        'dataset': dataset_name,
                        'method': name, 'n': n, 'rep': rep,
                        'time_s': round(time.time()-t0, 2),
                        'swag_ok': True, **met})
                except Exception as e:
                    print(f"  ERROR {name} n={n} rep={rep}: {e}")

            if (rep+1) % 10 == 0:
                elapsed = (time.time()-t_start)/60
                print(f"  rep {rep+1}/{N_REPEATS}  ({elapsed:.1f} min)",
                      flush=True)
                # ── 每10个rep保存一次checkpoint ──────────────
                pd.DataFrame(results).to_csv(ckpt_file, index=False)
                print(f"  checkpoint saved → {ckpt_file}", flush=True)

        # ── 每个n跑完立刻保存一次 ────────────────────────────
        pd.DataFrame(results).to_csv(output_file, index=False)
        print(f"  ✓ n={n} done, saved → {output_file}", flush=True)

    df = pd.DataFrame(results)
    df.to_csv(output_file, index=False)
    print(f"\n✓ COMPLETE: {output_file}  ({len(df)} rows)")

    print(f"\n── CRPS [{dataset_name}]")
    grp = df.groupby(['method','n'])['crps']
    mean_df = grp.mean().unstack().round(3)
    std_df  = grp.std().unstack().round(3)
    for col in mean_df.columns:
        mean_df[col] = (mean_df[col].astype(str) + " ±"
                        + std_df[col].astype(str))
    print(mean_df.to_string())

    print(f"\n── PICP [{dataset_name}]")
    print(df.groupby(['method','n'])['picp'].mean().unstack().round(3))

    sw = df[df.method=='SWAG'].groupby('n')['swag_ok'].agg(['sum','count'])
    sw['rate'] = (sw['sum']/sw['count']).round(2)
    print(f"\n── SWAG convergence [{dataset_name}]")
    print(sw.to_string())

    return df

# ── Run ───────────────────────────────────────────────────────
if __name__ == '__main__':
    t0 = time.time()

    print("Loading Kin8nm...")
    X_kin, y_kin = load_kin8nm()
    df_kin = run_dataset('kin8nm', X_kin, y_kin)

    print("\nLoading Naval...")
    X_nav, y_nav = load_naval()
    df_nav = run_dataset('naval', X_nav, y_nav)

    print(f"\nTotal runtime: {(time.time()-t0)/60:.1f} min")
    print("Files saved:")
    print("  uq_results_kin8nm.csv")
    print("  uq_results_naval.csv")
    print("  uq_checkpoint_kin8nm.csv  (intermediate)")
    print("  uq_checkpoint_naval.csv   (intermediate)")

In [ ]:
"""
BBB KL Weight Sensitivity
=========================
Only runs BBB at n=100, R=30, for three KL weights.
"""
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import properscoring as ps
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

# ── Config ────────────────────────────────────────────────────
N          = 100
N_REPEATS  = 30
EPOCHS     = 1000
LR         = 1e-3
HIDDEN     = 64
MC_SAMPLES = 50
BBB_KL_WEIGHTS = [1e-5, 1e-4, 1e-3]
SIGMA_FLOOR    = 0.05

# ── Synthetic data (same as main experiment) ──────────────────
def generate_synthetic(n_total=1200, seed=42):
    rng = np.random.RandomState(seed)
    X = rng.randn(n_total, 8).astype(np.float32)
    w = np.array([1.5, -2.0, 0.5, 1.0, -0.5, 0.3, -1.2, 0.8],
                 dtype=np.float32)
    sigma_x = 0.3 + 0.5 * np.abs(X[:, 0])
    y = X @ w + rng.randn(n_total).astype(np.float32) * sigma_x
    return X, y

X_all, y_all = generate_synthetic()
sc_X = StandardScaler().fit(X_all)
sc_y = StandardScaler().fit(y_all.reshape(-1,1))
X_all = sc_X.transform(X_all).astype(np.float32)
y_all = sc_y.transform(y_all.reshape(-1,1)).ravel().astype(np.float32)

n_test   = int(1200 * 0.30)
X_te     = torch.tensor(X_all[-n_test:])
y_te     = y_all[-n_test:]
X_pool   = X_all[:-n_test]
y_pool   = y_all[:-n_test]

# ── Model ─────────────────────────────────────────────────────
def gaussian_nll(out, y_t):
    mu, lv = out[:, 0], out[:, 1]
    var = torch.exp(lv).clamp(1e-3, 1e3)
    return (0.5 * ((y_t - mu)**2 / var + lv)).mean()

class BayesLinear(nn.Module):
    def __init__(self, in_f, out_f, prior_std=1.5):
        super().__init__()
        self.prior_std = prior_std
        self.wm = nn.Parameter(torch.empty(out_f, in_f).normal_(0, 0.1))
        self.wr = nn.Parameter(torch.full((out_f, in_f), -4.0))
        self.bm = nn.Parameter(torch.zeros(out_f))
        self.br = nn.Parameter(torch.full((out_f,), -4.0))
    def forward(self, x):
        ws = torch.log1p(torch.exp(self.wr)) + 1e-6
        bs = torch.log1p(torch.exp(self.br)) + 1e-6
        return nn.functional.linear(
            x, self.wm + ws * torch.randn_like(self.wm),
               self.bm + bs * torch.randn_like(self.bm))
    def kl(self):
        def _kl(mu, rho):
            s = torch.log1p(torch.exp(rho)) + 1e-6
            return 0.5 * ((s/self.prior_std)**2 + (mu/self.prior_std)**2
                          - 1 + 2*np.log(self.prior_std)
                          - 2*torch.log(s)).sum()
        return _kl(self.wm, self.wr) + _kl(self.bm, self.br)

class BayesMLP(nn.Module):
    def __init__(self, in_dim):
        super().__init__()
        self.l1  = BayesLinear(in_dim, HIDDEN)
        self.l2  = BayesLinear(HIDDEN,  HIDDEN)
        self.lo  = BayesLinear(HIDDEN,  2)
        self.act = nn.ReLU()
    def forward(self, x):
        return self.lo(self.act(self.l2(self.act(self.l1(x)))))
    def kl(self):
        return self.l1.kl() + self.l2.kl() + self.lo.kl()

# ── Run ───────────────────────────────────────────────────────
results = []

for kl_w in BBB_KL_WEIGHTS:
    crps_vals = []
    print(f"\nKL weight = {kl_w:.0e}")

    for rep in range(N_REPEATS):
        idx    = np.random.RandomState(rep + N).choice(
                     len(X_pool), N, replace=False)
        X_tr   = torch.tensor(X_pool[idx])
        y_tr   = torch.tensor(y_pool[idx])

        model  = BayesMLP(8)
        opt    = optim.Adam(model.parameters(), lr=LR)

        model.train()
        for _ in range(EPOCHS):
            opt.zero_grad()
            loss = gaussian_nll(model(X_tr), y_tr) + kl_w * model.kl()
            loss.backward()
            opt.step()

        model.train()
        with torch.no_grad():
            outs = torch.stack([model(X_te) for _ in range(MC_SAMPLES)])
        mu    = outs[..., 0].mean(0).numpy()
        sigma = torch.sqrt(outs[..., 0].var(0) +
                           torch.exp(outs[..., 1]).mean(0)).numpy()
        sigma = np.clip(sigma, SIGMA_FLOOR, None)
        crps  = float(ps.crps_gaussian(y_te, mu, sigma).mean())
        crps_vals.append(crps)

        if (rep+1) % 10 == 0:
            print(f"  rep {rep+1}/{N_REPEATS}  "
                  f"running mean CRPS = {np.mean(crps_vals):.4f}")

    mean_crps = round(np.mean(crps_vals), 3)
    std_crps  = round(np.std(crps_vals),  3)
    print(f"  → Mean CRPS = {mean_crps} ± {std_crps}")
    results.append({'kl_weight': kl_w,
                    'mean_crps': mean_crps,
                    'std_crps':  std_crps})

# ── Summary ───────────────────────────────────────────────────
print("\n=== BBB Sensitivity Results ===")
df = pd.DataFrame(results)
print(df.to_string(index=False))
df.to_csv("bbb_sensitivity.csv", index=False)
print("\nSaved → bbb_sensitivity.csv")
print("\nFor LaTeX table:")
for _, row in df.iterrows():
    exp = int(np.log10(row.kl_weight))
    print(f"  $10^{{{exp}}}$ & {row.mean_crps} \\\\")

In [ ]:
# ═══════════════════════════════════════════════════════════════
# R code for BHM final — Method-specific sigma, conditional SWAG inclusion 4.4
#
# SWAG included only when convergence rate = 1.00:
#   Synthetic: n=200, 500
#   Energy:    none (max convergence rate = 0.94)
#   Concrete:  n=100, 200, 500
#   Kin8nm:    n=200, 500  (n=100 rate=0.98, borderline)
#   Yacht:     none (max rate=0.98 but n=100 rate=0.96)
#
# Input:  uq_final_results.csv
#         uq_results_real_combined.csv
#         uq_results_kin8nm.csv
#         uq_results_yacht.csv
# Output: bhm_pairwise_final.csv
#         bhm_real_pairwise_final.csv
#         p_mcd_ensemble_final.csv
# ═══════════════════════════════════════════════════════════════

library(tidyverse)
library(brms)

set.seed(42)

# ── SWAG convergence rates (from experiments) ─────────────────
# Only include SWAG when rate = 1.00
SWAG_OK <- list(
  synthetic = c(200, 500),
  energy    = c(),           # max rate 0.94, never 1.00
  concrete  = c(100, 200, 500),
  kin8nm    = c(200, 500),   # n=100 rate=0.98, exclude
  yacht     = c()            # max rate 0.98, exclude all
)

# ── BHM fitting function ──────────────────────────────────────
fit_bhm <- function(d, dataset, n_val) {
  # Conditional SWAG inclusion
  swag_ok_ns <- SWAG_OK[[dataset]]
  if (is.null(swag_ok_ns) || !n_val %in% swag_ok_ns) {
    d <- d %>% filter(method != "SWAG")
  }
  
  if (nrow(d) < 10 || length(unique(d$method)) < 2) return(NULL)
  
  brm(
    bf(value ~ 0 + method,
       sigma ~ 0 + method),
    data   = d,
    family = gaussian(),
    prior  = c(
      prior(normal(0, 2), class = b),
      prior(normal(0, 1), class = b, dpar = sigma)
    ),
    chains = 2, iter = 2000, warmup = 1000,
    cores  = 1, refresh = 0, silent = 2
  )
}

# ── Extract pairwise probabilities ────────────────────────────
extract_pairs <- function(fit, metric, n_val, dataset) {
  if (is.null(fit)) return(NULL)
  
  draws        <- posterior::as_draws_df(fit)
  method_cols  <- names(draws)[grepl("^b_method", names(draws)) &
                                 !grepl("sigma", names(draws))]
  method_names <- gsub("^b_method", "", method_cols)
  
  pairs <- expand_grid(A = method_names, B = method_names) %>%
    filter(A != B) %>%
    rowwise() %>%
    mutate(
      col_A         = paste0("b_method", A),
      col_B         = paste0("b_method", B),
      prob_A_better = mean(draws[[col_A]] < draws[[col_B]]),
      mdd_80        = 1.28 * sd(draws[[col_A]] - draws[[col_B]])
    ) %>%
    ungroup() %>%
    select(-col_A, -col_B) %>%
    mutate(metric = metric, n = n_val, dataset = dataset)
  
  pairs
}

# ── Load data ─────────────────────────────────────────────────
cat("Loading data...\n")

synth_raw <- read_csv("uq_final_results.csv",
                      show_col_types = FALSE) %>%
  filter(!(method == "SWAG" & swag_ok == FALSE)) %>%
  mutate(method = factor(method),
         n_num  = as.integer(as.character(n)))

real_raw <- bind_rows(
  read_csv("uq_results_real_combined.csv",
           show_col_types = FALSE),
  read_csv("uq_results_kin8nm.csv",
           show_col_types = FALSE) %>% mutate(dataset = "kin8nm"),
  read_csv("uq_results_yacht.csv",
           show_col_types = FALSE) %>% mutate(dataset = "yacht")
) %>%
  filter(!(method == "SWAG" & swag_ok == FALSE)) %>%
  mutate(method  = factor(method),
         n_num   = as.integer(as.character(n)),
         dataset = as.character(dataset))

cat(sprintf("Synthetic: %d rows\n", nrow(synth_raw)))
cat(sprintf("Real: %d rows, datasets: %s\n",
            nrow(real_raw),
            paste(unique(real_raw$dataset), collapse=", ")))

# ══════════════════════════════════════════════════════════════
# SYNTHETIC
# ══════════════════════════════════════════════════════════════
TARGET_METRICS <- c("crps", "picp", "interval_score")
N_VALS         <- c(30, 50, 100, 200, 500)
synth_results  <- list()

cat("\n=== SYNTHETIC ===\n")
for (metric in TARGET_METRICS) {
  cat(sprintf("  %-18s: ", metric))
  for (n_val in N_VALS) {
    swag_in <- n_val %in% SWAG_OK[["synthetic"]]
    cat(sprintf("n=%d%s ", n_val, ifelse(swag_in,"(+S)","")),
        flush = TRUE)
    
    d <- synth_raw %>%
      filter(n_num == n_val, !is.na(.data[[metric]])) %>%
      select(method, value = all_of(metric))
    
    fit   <- fit_bhm(d, "synthetic", n_val)
    pairs <- extract_pairs(fit, metric, n_val, "synthetic")
    if (!is.null(pairs))
      synth_results[[paste("synthetic", metric, n_val)]] <- pairs
  }
  cat("\n")
}

synth_df <- bind_rows(synth_results)
write_csv(synth_df, "bhm_pairwise_final.csv")
cat(sprintf("Saved bhm_pairwise_final.csv (%d rows)\n", nrow(synth_df)))

# ══════════════════════════════════════════════════════════════
# REAL DATASETS
# ══════════════════════════════════════════════════════════════
DATASETS     <- setdiff(unique(real_raw$dataset), "synthetic")
real_results <- list()

for (dset in DATASETS) {
  df_sub <- real_raw %>% filter(dataset == dset)
  n_vals <- sort(unique(df_sub$n_num))
  cat(sprintf("\n=== %s ===\n", toupper(dset)))
  
  for (metric in TARGET_METRICS) {
    cat(sprintf("  %-18s: ", metric))
    for (n_val in n_vals) {
      swag_in <- n_val %in% SWAG_OK[[dset]]
      cat(sprintf("n=%d%s ", n_val,
                  ifelse(swag_in,"(+S)","")), flush = TRUE)
      
      d <- df_sub %>%
        filter(n_num == n_val, !is.na(.data[[metric]])) %>%
        select(method, value = all_of(metric))
      
      fit   <- fit_bhm(d, dset, n_val)
      pairs <- extract_pairs(fit, metric, n_val, dset)
      if (!is.null(pairs))
        real_results[[paste(dset, metric, n_val)]] <- pairs
    }
    cat("\n")
  }
}

real_df <- bind_rows(real_results)
write_csv(real_df, "bhm_real_pairwise_final.csv")
cat(sprintf("\nSaved bhm_real_pairwise_final.csv (%d rows)\n",
            nrow(real_df)))

# ══════════════════════════════════════════════════════════════
# SUMMARY TABLES
# ══════════════════════════════════════════════════════════════
all_df <- bind_rows(synth_df, real_df)

cat("\n=== P(MCD < Ensemble) on CRPS ===\n")
p_table <- all_df %>%
  filter(metric == "crps", A == "MCD", B == "Ensemble") %>%
  select(dataset, n, prob_A_better) %>%
  mutate(prob_A_better = round(prob_A_better, 3)) %>%
  arrange(dataset, n) %>%
  pivot_wider(names_from  = n,
              values_from = prob_A_better,
              names_prefix = "n=") %>%
  mutate(dataset = factor(dataset,
                          levels = c("synthetic","energy","concrete","kin8nm","yacht"))) %>%
  arrange(dataset)

print(p_table, width = Inf)
write_csv(p_table, "p_mcd_ensemble_final.csv")

cat("\n=== MDD (CRPS, MCD vs Ensemble) ===\n")
all_df %>%
  filter(metric == "crps", A == "MCD", B == "Ensemble") %>%
  select(dataset, n, mdd_80) %>%
  mutate(mdd_80 = round(mdd_80, 4)) %>%
  pivot_wider(names_from  = n,
              values_from = mdd_80,
              names_prefix = "n=") %>%
  mutate(dataset = factor(dataset,
                          levels = c("synthetic","energy","concrete","kin8nm","yacht"))) %>%
  arrange(dataset) %>%
  print(width = Inf)

cat("\n=== P(MCD < MAP) on CRPS ===\n")
all_df %>%
  filter(metric == "crps", A == "MCD", B == "MAP") %>%
  select(dataset, n, prob_A_better) %>%
  mutate(prob_A_better = round(prob_A_better, 3)) %>%
  pivot_wider(names_from  = n,
              values_from = prob_A_better,
              names_prefix = "n=") %>%
  mutate(dataset = factor(dataset,
                          levels = c("synthetic","energy","concrete","kin8nm","yacht"))) %>%
  arrange(dataset) %>%
  print(width = Inf)

cat("\n=== Done ===\n")
cat("Output files:\n")
cat("  bhm_pairwise_final.csv\n")
cat("  bhm_real_pairwise_final.csv\n")
cat("  p_mcd_ensemble_final.csv\n")

In [ ]:
## R code for figure 3 heatmap
## 如果可以的话想看一个包含SWAG的版本的图
library(tidyverse)
library(patchwork)

METHOD_LEVELS <- c("MAP", "CP", "BBB", "Ensemble", "MCD")

# ── Load data ─────────────────────────────────────────────────
synth <- read_csv("bhm_pairwise_final.csv", show_col_types = FALSE)
real  <- read_csv("bhm_real_pairwise_final.csv", show_col_types = FALSE)

all_df <- bind_rows(synth, real) %>%
    filter(
        metric == "crps",
        n      == 50,
        A %in% METHOD_LEVELS,
        B %in% METHOD_LEVELS
    ) %>%
    mutate(
        A = factor(A, levels = rev(METHOD_LEVELS)),
        B = factor(B, levels = METHOD_LEVELS)
    )

# ── Plot function ─────────────────────────────────────────────
plot_heatmap <- function(data, title, show_legend = FALSE) {
    ggplot(data, aes(x = B, y = A)) +
        geom_tile(
            data = data %>%
                filter(as.character(A) == as.character(B)),
            fill = "grey85", color = "white", linewidth = 0.5
        ) +
        geom_tile(
            data = data %>%
                filter(as.character(A) != as.character(B)),
            aes(fill = prob_A_better),
            color = "white", linewidth = 0.5
        ) +
        geom_text(
            data = data %>%
                filter(as.character(A) != as.character(B)),
            aes(label = sprintf("%.2f", prob_A_better)),
            size = 3.8, color = "black", fontface = "bold"
        ) +
        scale_fill_gradient2(
            low      = "#d62728",
            mid      = "white",
            high     = "#1f77b4",
            midpoint = 0.5,
            limits   = c(0, 1),
            name     = expression(P(A %<% B)),
            na.value = "grey85"
        ) +
        labs(title = title, x = "Method B", y = "Method A") +
        theme_bw(base_size = 11) +
        theme(
            axis.text.x     = element_text(angle = 30, hjust = 1,
                                           size = 10),
            axis.text.y     = element_text(size = 10),
            axis.title      = element_text(face = "bold", size = 11),
            panel.grid      = element_blank(),
            panel.border    = element_rect(color = "black", fill = NA,
                                           linewidth = 0.8),
            plot.title      = element_text(face = "bold", hjust = 0.5,
                                           size = 12),
            legend.position = if (show_legend) "right" else "none"
        )
}

# ── Build panels ──────────────────────────────────────────────
p1 <- plot_heatmap(
    all_df %>% filter(dataset == "kin8nm"),
    title       = "Kin8nm  (n = 50)",
    show_legend = FALSE
)

p2 <- plot_heatmap(
    all_df %>% filter(dataset == "concrete"),
    title       = "Concrete  (n = 50)",
    show_legend = TRUE
)

# ── Combine ───────────────────────────────────────────────────
fig4 <- p1 + p2 +
    plot_layout(ncol = 2, widths = c(1, 1.25)) +
    plot_annotation(
        title = "Pairwise Posterior Probabilities at n = 50",
        theme = theme(
            plot.title = element_text(face = "bold", hjust = 0.5,
                                      size = 14)
        )
    )

# ── Save ──────────────────────────────────────────────────────
ggsave("figures/fig4_heatmap.pdf", fig4,
       width = 10, height = 4.8, device = cairo_pdf)
ggsave("figures/fig4_heatmap.png", fig4,
       width = 10, height = 4.8, dpi = 150)
cat("Saved fig4_heatmap.pdf and fig4_heatmap.png\n")

In [ ]:
# R code for figure 5 Model fit.

library(brms)
library(tidyverse)

raw <- read_csv("uq_final_results.csv", show_col_types = FALSE)
df  <- raw %>%
    filter(!(method == "SWAG" & swag_ok == FALSE)) %>%
    mutate(method = factor(method),
           n_num  = as.integer(as.character(n)))

# 用n=50, CRPS作为representative fit
d_sub <- df %>%
    filter(n_num == 50, method != "SWAG") %>%
    select(method, value = crps)

fit_ppc <- brm(
    bf(value ~ 0 + method,
       sigma ~ 0 + method),
    data   = d_sub,
    family = gaussian(),
    prior  = c(prior(normal(0, 2), class = b),
               prior(normal(0, 1), class = b, dpar = sigma)),
    chains = 2, iter = 2000, warmup = 1000,
    cores  = 1, refresh = 0, silent = 2
)

p <- pp_check(fit_ppc, ndraws = 100) +
    labs(
        title = NULL,
        x     = "CRPS",
        y     = "Density"
    ) +
    theme_bw(base_size = 12) +
    theme(
        panel.grid.minor = element_blank(),
        legend.position  = c(0.95, 0.95),
        legend.justification = c(1, 1)
    )

ggsave("figures/ppc_crps_n50.pdf", p,
       width = 6, height = 4, device = cairo_pdf)
ggsave("figures/ppc_crps_n50.png", p,
       width = 6, height = 4, dpi = 150)

In [ ]:
# python code for Figure 6: Variance decomposition 
# load variance_results.npz in google cloude

import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import linregress

data = np.load("variance_results.npz")
n_values = data['n']
total_var = data['total']
algo_var = data['algo']
data_var = data['data']

alpha_total = -linregress(np.log(n_values), np.log(total_var)).slope
alpha_algo = -linregress(np.log(n_values), np.log(algo_var)).slope
alpha_data = -linregress(np.log(n_values), np.log(data_var)).slope

plt.rcParams.update({
    "font.size": 12,
    "axes.labelsize": 14,
    "axes.titlesize": 16,
    "legend.fontsize": 12,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "font.family": "serif",
})

fig, ax = plt.subplots(figsize=(8, 6))

ax.loglog(n_values, total_var, marker='o', linestyle='-', linewidth=2.5, markersize=8,
          label=rf"Total Variance ($\alpha={alpha_total:.2f}$)")
ax.loglog(n_values, algo_var, marker='s', linestyle='--', linewidth=2.5, markersize=8,
          label=rf"Algorithm Variance ($\alpha={alpha_algo:.2f}$)")
ax.loglog(n_values, data_var, marker='^', linestyle='-.', linewidth=2.5, markersize=8,
          label=rf"Data Variance ($\alpha={alpha_data:.2f}$)")
ax.loglog(n_values, (1/n_values) * (total_var[0] * n_values[0]), linestyle=':', linewidth=2,
          label=r"Theoretical $O(n^{-1})$")

ax.set_xlabel(r"Training Size $n$")
ax.set_ylabel("Variance")
ax.set_title("Variance Decomposition in BDL Evaluation", pad=15)
ax.grid(True, which="major", linestyle="-", linewidth=0.8, alpha=0.3)
ax.grid(True, which="minor", linestyle=":", linewidth=0.5, alpha=0.2)
ax.legend(loc="best", frameon=True, fancybox=False, edgecolor='black', framealpha=0.8)

fig.tight_layout()
plt.savefig("variance_decomposition.pdf", format="pdf", bbox_inches="tight")
plt.savefig("variance_decomposition.png", format="png", bbox_inches="tight", dpi=300)
plt.show()

print(f"Alpha total: {alpha_total:.4f}")
print(f"Alpha algorithm: {alpha_algo:.4f}")
print(f"Alpha data: {alpha_data:.4f}")

In [ ]:
## 4.6 update for predicted MDD using posterior predictive distribution
# input : uq_final_results.csv

compute_mdd_correct <- function(fit, method_A, method_B, gamma = 0.80) {
    
    draws <- posterior::as_draws_df(fit)
    z_gamma <- qnorm(gamma)
    
    # Mean posterior draws
    mu_A <- draws[[paste0("b_method", method_A)]]
    mu_B <- draws[[paste0("b_method", method_B)]]
    
    # Robust sigma column detection
    sigma_col_A <- grep(
        paste0("^b_sigma_method", method_A, "$"),
        names(draws),
        value = TRUE
    )
    
    sigma_col_B <- grep(
        paste0("^b_sigma_method", method_B, "$"),
        names(draws),
        value = TRUE
    )
    
    # Convert from log scale
    sigma_A <- exp(draws[[sigma_col_A]])
    sigma_B <- exp(draws[[sigma_col_B]])
    
    # Posterior predictive difference
    diff_pred <- (mu_A - mu_B) +
        rnorm(
            length(mu_A),
            mean = 0,
            sd = sqrt(sigma_A^2 + sigma_B^2)
        )
    
    # Correct predictive MDD
    sigma_diff_pred <- sd(diff_pred)
    mdd_correct <- z_gamma * sigma_diff_pred
    
    # Mean-only version (original)
    sigma_diff_mean <- sd(mu_A - mu_B)
    mdd_incorrect <- z_gamma * sigma_diff_mean
    
    list(
        mdd_correct = mdd_correct,
        mdd_incorrect = mdd_incorrect,
        ratio = mdd_correct / mdd_incorrect,
        sigma_pred = sigma_diff_pred,
        sigma_mean = sigma_diff_mean,
        p_MCD_better = mean(mu_A - mu_B < 0),
        p_pred_better = mean(diff_pred < 0)
    )
}

# ═══════════════════════════════════════════════════════════════
# Run for each n
# ═══════════════════════════════════════════════════════════════

results_correct <- list()

for (n_val in N_VALS) {
    
    cat(sprintf("\nn = %d\n", n_val))
    
    # Ensure value column exists
    d <- raw %>%
        filter(
            n_num == n_val,
            !is.na(crps),
            method != "SWAG"
        ) %>%
        filter(method %in% METHOD_LEVELS) %>%
        select(
            method,
            rep,
            value = crps
        )
    
    cat("  Fitting hierarchical model...\n")
    
    fit <- brm(
        bf(
            value ~ 0 + method,
            sigma ~ 0 + method
        ),
        data = d,
        family = gaussian(),
        prior = c(
            prior(normal(0, 2), class = b),
            prior(normal(0, 1), class = b, dpar = sigma)
        ),
        chains = 2,
        iter = 2000,
        warmup = 1000,
        cores = 1,
        refresh = 0,
        silent = 2
    )
    
    res <- compute_mdd_correct(
        fit,
        "MCD",
        "Ensemble",
        gamma = 0.80
    )
    
    results_correct[[as.character(n_val)]] <- tibble(
        n = n_val,
        mdd_correct = round(res$mdd_correct, 4),
        mdd_incorrect = round(res$mdd_incorrect, 4),
        ratio = round(res$ratio, 2),
        p_MCD_better = round(res$p_MCD_better, 3),
        p_pred_better = round(res$p_pred_better, 3)
    )
    
    cat(sprintf("  MDD (mean-only):     %.4f\n", res$mdd_incorrect))
    cat(sprintf("  MDD (predictive):    %.4f\n", res$mdd_correct))
    cat(sprintf("  Ratio:               %.2fx\n", res$ratio))
    cat(sprintf("  P(mean better):      %.3f\n", res$p_MCD_better))
    cat(sprintf("  P(new rep better):   %.3f\n", res$p_pred_better))
}

out_correct <- bind_rows(results_correct)

print(out_correct, width = Inf)

In [ ]:
# figure 5, need revise, consider MCD vs MAP or Ensemble vs MAP, see which is the best for selling
# ═══════════════════════════════════════════════════════════════════════════
# Figure 5: Predictive Minimum Detectable Difference (MCD vs Ensemble on CRPS)
# ═══════════════════════════════════════════════════════════════════════════

library(tidyverse)
library(cowplot)

results <- tibble(
    n = c(30, 50, 100, 200, 500),
    mdd_correct = c(0.0458, 0.0211, 0.0169, 0.0095, 0.0054),  # Predictive MDD
    observed_diff = c(0.0166, 0.0146, 0.0168, 0.0247, 0.0214),  # |μ_MCD - μ_Ensemble|
    p_mean_better = c(0.014, 1.000, 1.000, 1.000, 1.000),  # From Figure 2
    p_pred_better = c(0.370, 0.717, 0.803, 0.987, 1.000)   # Probability new rep detects
)

p_mdd <- results %>%
    mutate(
        detectable = observed_diff > mdd_correct,
        detectable_label = ifelse(detectable, "Detectable", "Not detectable"),
        y_min = 0,
        y_max = ifelse(detectable, NA, mdd_correct)  # Shade only undetectable region
    ) %>%
    ggplot(aes(x = n)) +
    
    # Shaded region for "undetectable" (observed diff < MDD)
    geom_ribbon(
        data = . %>% filter(!detectable),
        aes(ymin = 0, ymax = mdd_correct, fill = "Not detectable"),
        alpha = 0.25
    ) +
    
    # Shaded region for "detectable" (observed diff > MDD)
    geom_ribbon(
        data = . %>% filter(detectable),
        aes(ymin = 0, ymax = observed_diff, fill = "Detectable"),
        alpha = 0.15
    ) +
    
    # MDD line (red)
    geom_line(aes(y = mdd_correct, color = "MDD (predictive)"), 
              size = 1.2, linetype = "solid") +
    geom_point(aes(y = mdd_correct, color = "MDD (predictive)"), 
               size = 3) +
    
    # Observed difference line (blue, dashed)
    geom_line(aes(y = observed_diff, color = "Observed |Δμ|"), 
              size = 1.2, linetype = "dashed") +
    geom_point(aes(y = observed_diff, color = "Observed |Δμ|"), 
               size = 3) +
    
    # Cross-over point annotation
    geom_vline(xintercept = 200, linetype = "dotted", alpha = 0.5) +
    annotate("text", x = 220, y = 0.035, 
             label = "Cross-over at n = 200", 
             size = 3, hjust = 0, alpha = 0.7) +
    
    # Scales
    scale_x_log10(
        breaks = c(30, 50, 100, 200, 500),
        labels = c("30", "50", "100", "200", "500")
    ) +
    scale_y_continuous(
        limits = c(0, 0.055),
        breaks = seq(0, 0.05, by = 0.01),
        labels = seq(0, 0.05, by = 0.01)
    ) +
    scale_color_manual(
        name = "",
        values = c("MDD (predictive)" = "#D55E00",  # Red-orange
                   "Observed |Δμ|" = "#0072B2")     # Blue
    ) +
    scale_fill_manual(
        name = "Detection status",
        values = c("Detectable" = "#009E73",      # Green
                   "Not detectable" = "#CC79A7")  # Purple-pink
    ) +
    
    # Labels
    labs(
        x = expression("Training set size" ~ italic(n)),
        y = expression("CRPS difference" ~ (|Delta*CRPS|)),
        title = NULL,
        subtitle = NULL
    ) +
    
    # Theme
    theme_bw(base_size = 11) +
    theme(
        legend.position = c(0.72, 0.85),
        legend.background = element_rect(fill = "white", size = 0.3),
        legend.key.size = unit(0.4, "cm"),
        legend.spacing.y = unit(0.1, "cm"),
        panel.grid.minor = element_blank(),
        plot.margin = margin(t = 5, r = 10, b = 5, l = 5)
    )

print(p_mdd)
ggsave("figure5_mdd.pdf", p_mdd, width = 4.5, height = 3.5)
ggsave("figure5_mdd.png", p_mdd, width = 4.5, height = 3.5, dpi = 300)